# Flood Risk Assessment
## Building and Agriculture Damage and Population Exposure and Displaced

- A workflow from the CLIMAAX [Handbook](https://handbook.climaax.eu/) and [FLOODS](https://github.com/CLIMAAX/FLOODS) GitHub repository.
- See our [how to use risk workflows](https://handbook.climaax.eu/notebooks/workflows_how_to.html) page for information on how to run this notebook.

## Introduction
This workflow assesses economic damage to buildings, exposure of critical infrastructure, and population exposure & displacement, by combining flood map data (hazard) and population and building data (exposure).

<figure class="align-center">
  <iframe width="560" height="315" src="https://www.youtube-nocookie.com/embed/JWQbkcf75T4?si=CaBLEpgdI90EHOmE" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>
</figure>

### Building and Agriculture Damage & Critical Infrastructure
Flood risk in the form of economic damage at a building level is computed from flood depth maps and building data. The damages are computed for each flood event (flood map for a given return period) and then integrated over all of the event return periods to determine expected annual damage (EAD). Damage to each building varies based on the local flood depth, reconstruction costs, value of its contents, and its footprint area.

Datasets:
- River flood extent and depth are from the [European Commission's Joint Research Centre Data Catalogue](https://data.jrc.ec.europa.eu/collection/id-0054) for two datasets at different return periods (10- to 500-year) on a 3 arcsecond (\~90 m) grid. In addition, river (and coastal) flood data are available from the World Resource Institute (WRI) product called [Aqueduct Floods](https://www.wri.org/applications/aqueduct/floods) for the historical climate as well as for the RCP 4.5 and 8.5 projections for the 2030s, 2050s and 2080s and for five global climate models for different return periods (10- to 1000-year) at a 30 arcsecond (\~925 km) resolution accessed using [Google Earth Engine](https://developers.google.com/earth-engine/datasets/catalog/WRI_Aqueduct_Flood_Hazard_Maps_V2).
  - [European Level](https://data.jrc.ec.europa.eu/dataset/1d128b6c-a4ee-4858-9e34-6210707f3c81)
  - [Global Level](https://data.jrc.ec.europa.eu/dataset/jrc-floods-floodmapgl_rp50y-tif)
  - [Aqueduct Floods](https://wri-projects.s3.amazonaws.com/AqueductFloodTool/download/v2/index.html)
- Coastal flood extent and depth with and wihtout subsidence are available from WRI [Aqueduct Floods](https://www.wri.org/applications/aqueduct/floods) for the historical climate as well as for the RCP 4.5 and 8.5 projections for the 2030s, 2050s and 2080s and for five global climate models accessed using [Google Earth Engine](https://developers.google.com/earth-engine/datasets/catalog/WRI_Aqueduct_Flood_Hazard_Maps_V2).
- Building data, including type and footprint are obtained from [OpenStreetMap](https://www.openstreetmap.org/copyright).
- Building damage fraction, reconstruction costs, and the value of its contents are determined using the [JRC methodology](https://publications.jrc.ec.europa.eu/repository/handle/JRC105688).
- Landuse Data are from the Global Change Analysis Model - Demeter Landuse [GCAM-Demeter-LU](https://data.pnnl.gov/group/nodes/dataset/13192) several SSP-RCP combinations at a 0.05-degree resolution and time periods from 2015 to 2100 using a GCM ensemble average.

Furthermore, critical infrastructure is mapped onto the flood maps to visually assess its exposure to the hazard.

The code can be customized to use any flood map, building data, and depth-damage relationships.

### Population exposure & population displaced
Population exposure and displaced population are computed from flood maps and population maps. The population data by default is based on a global dataset and one can choose from a number of options for the time period (past years and projections for near future). Population is considered displaced when the population is exposed to flood depths over a given threshold. For both exposure and displacement, the results are integrated over all of the event return periods to determine expected annual population exposed (EAPE) and expected annual population displaced (EAPD).

Datasets:

- Population data are available for the [Global Human Settlement Layer (GHSL)](https://data.jrc.ec.europa.eu/dataset/2ff68a52-5b5b-4a22-8f40-c41da8332cfe) and the NASA Socioeconomic Data and Applications Center (SEDAC) [Global 1-km Downscaled Population Projection Grids](https://beta.sedac.ciesin.columbia.edu/data/set/popdynamics-pop-projection-ssp-downscaled-1km-2010-2100/data-download) for the five SSPs over the period 2010 to 2100.
- [GCAM-Demeter-LU](https://data.pnnl.gov/group/nodes/dataset/13192) landuse projection are available for several SSP-RCP combinations from 2015 to 2100 at a 0.05-degree resolution. For simplification, all crop types are summed into a single category and the model enemble average is used.


The code can be customized to use any population data, flood map, and minimum water depth threshold to classify the exposed and displaced populations.

### Limitations
The flood maps that are used in this workflow by default do not take into consideration possible flood protection infrastructure that may in reality limit the impact of the hazard. Moreover, the resolution of 3 arc-seconds for both the flood maps and population maps may be unsitable for particularly complex regions. If possible, it is suggested to use local data which may lead to a better representation of the ground truth.

Furthermore, buildings that are close to water bodies (e.g: rivers) may overlap the water body, thus resulting in higher than expected flood depths (and damage) values being used in the damage calculations. This effect can be reduced by using higher resolution flood maps.
Similarly, due to the resolution of both the population and the flood maps, it might be that part of the population appears to be over a water body and is counted towards the overall exposed and displaced statistics irregardless of flooding.

## Preparation work
### Import modules

`````{admonition} Find more info about the libraries used in this workflow here
:class: hint dropdown

These modules are needed to process the data in this workflow and to plot the results.
- [os](https://docs.python.org/3/library/os.html): For interacting with the operating system, allowing the creation of directories and file manipulation.
- [sys](https://docs.python.org/3/library/sys.html): Provides access to some variables used or maintained by the interpreter and to functions that interact strongly with the interpreter. It is always available.
- [numpy](https://numpy.org/): A powerful library for numerical computations in Python, widely used for array operations and mathematical functions.
- [pandas](https://pandas.pydata.org/): A data manipulation and analysis library, essential for working with structured data in tabular form.
- [geopandas](https://geopandas.org): Extends the datatypes used by pandas to allow spatial operations on geometric types.
- [rasterio](https://rasterio.readthedocs.io/en/stable/): For reading and writing geospatial raster data, providing functionalities to explore and manipulate raster datasets.
- [rasterstats](https://pythonhosted.org/rasterstats): For summarizing geospatial raster datasets based on vector geometries.
- [shapely](https://pypi.org/project/shapely/): For manipulation and analysis of planar geometric objects.
- [osgeo](https://www.osgeo.org/): For translating raster and vector geospatial data formats.
- [osmnx](https://osmnx.readthedocs.io/) To easily download, model, analyze, and visualize street networks and other geospatial features from OpenStreetMap.
- [pyproj](https://pyproj4.github.io/pyproj/dev/index.html): Interface for PROJ (cartographic projections and coordinate transformations library).
- [matplotlib](https://matplotlib.org/): Used for creating static, animated, and interactive visualizations.
- [contextily](https://contextily.readthedocs.io/en/latest/): For adding basemaps to plots, enhancing geospatial visualizations.
- [urllib.request](https://docs.python.org/3/library/urllib.request.html): Defines functions and classes which help in opening URLs (mostly HTTP) in a complex world — basic and digest authentication, redirections, cookies and more.
- [zipfile](https://docs.python.org/3/library/zipfile.html): Provides tools to create, read, write, append, and list a ZIP file.
- [socket](https://docs.python.org/3/library/socket.html): pPovides access to the BSD socket interface.

`````

In [ ]:
# Colab
#!apt-get update
#!apt-get install -y gdal-bin libgdal-dev proj-bin
#!pip install rioxarray netcdf4 h5netcdf rasterio rasterstats osmnx contextily geemap
#print('\nDone install packages')

# Local Machines:
#conda create -n conda_climaaxflood -c conda-forge python=3.10 numpy pandas geopandas xarray netcdf4 h5netcdf rioaxarry rasterio gdal osmnx pyproj geemap matplotlib contextily requests tqdm rasterstats jupyter

In [ ]:
import os
import sys
import urllib.request
from zipfile import ZipFile
import socket
import warnings
import numpy as np
import math
import pandas as pd
import geopandas as gpd
import geemap
import ee
import xarray as xr
import netCDF4
import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from rasterio.warp import reproject, calculate_default_transform
from rasterio.mask import mask
from rasterio.transform import array_bounds
from rasterio.merge import merge
import rasterstats
import rioxarray as rio
from osgeo import gdal, osr
from shapely.geometry import Polygon, Point, box
import osmnx as ox
from pyproj import Transformer
from affine import Affine
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from mpl_toolkits.axes_grid1 import make_axes_locatable
import contextily as ctx
import requests
from tqdm import tqdm  # Progress bars
print('Done loading packages')

### Define inputs
In this section different input parameters are set. Please read the descriptions when provided to ensure the appropriate changes are made.
Some of the things that can be set here are:
- Geographical bounds of area of interest (a .shp, .geojson or .gpkg file can be used for polygons delineating the area of interest, alternatively coordinates can be manually inserted for a rectangular outline).
- Return periods for which calculations are made, projection of the population.
- Code flags (e.g. choosing which maps get produced, what elements are shown in outputs, automatically saving images, etc.)
- Directory locations
- Colorbars for maps

In [ ]:
# Directory of main folder
dir_work = '/content'

# Prefex for file names and folder names containing computed data
file_prefix = 'NewOrleans'

# Choose flood map dataset: 'EU' or 'Global'
flood_datas = ['JRC-EU', 'JRC-Global', 'WRI']
flood_data = 'WRI'
if flood_data not in flood_datas:
    raise ValueError(f'ERROR: {flood_data} must be one of: {flood_datas}')
if flood_data == 'WRI':
    # Models: WATCH, NorESM1-M, GFDL_ESM2M, HadGEM2-ES, IPSL-CM5A-LR
    model = 'WATCH'
    # Scenarios: historical, rcp4p5, rcp4p5
    scenario = 'rcp4p5'
    # year: 2030, 2050, 2080
    year_scen = 2080
    # Type: river, coast
    flood_type = 'river'
    if flood_type == 'coast':
       # SLR Scenario 5 (low), 50 (median), 95 (high)
       slr_scenario = 50
       # Subsidence: wtsub (included), nosub (not included)
       subsidence	= 'wtsub'
    # Resolution of download GEE raster (WRI native resolution is 1000m)
    flood_rast_res = 100

# Landuse SSP & Year
lu_ssps = ['SSP126', 'SSP145', 'SSP160',
           'SSP226', 'SSP245', 'SSP260',
           'SSP345', 'SSP360',
           'SSP426', 'SSP445', 'SSP460',
           'SSP526', 'SSP545', 'SSP560', 'SSP585']
lu_ssp = 'SSP585'
lu_year = 2100
# Years must be a multiple of 5 and between 2015 and 2100.
if lu_ssp not in lu_ssps:
    raise ValueError(f'ERROR: {lu_ssp} must be one of: {lu_ssps}')
if not (2015 <= lu_year <= 2100 and lu_year % 5 == 0):
    raise ValueError(f'ERROR: {pop_data} {lu_year} must be a multiple of 5 and between 2015 and 2100.')

# Population data options: 'GHSL' or 'SEDAC'
pop_datas = ['GHSL', 'SEDAC']
pop_data = 'GHSL'
if pop_data not in pop_datas:
    raise ValueError(f'ERROR: {pop_data} must be one of: {pop_datas}')
# Population Year
pop_year = 2030
# SEDAC Population SSP
if pop_data == 'SEDAC':
    pop_ssps = ['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']
    pop_ssp = 'SSP5'
    if pop_ssp not in pop_ssps:
        raise ValueError(f'ERROR: {pop_ssp} must be one of: {pop_ssps}')
    if not (2010 <= pop_year <= 2100 and pop_year % 10 == 0):
        raise ValueError(f'ERROR: {pop_year} must be a multiple of 10 and between 2010 and 2100.')
    file_pop_string = f'{pop_data}_{pop_ssp}_{pop_year}'
    title_pop_string = f'{pop_data} {pop_ssp} {pop_year}'
elif pop_data == 'GHSL':
    if not (1975 <= pop_year <= 2030 and pop_year % 5 == 0):
        raise ValueError(f'ERROR: {pop_data} {pop_year} must be a multiple of 5 and between 1975 and 2100.')
    file_pop_string = f'{pop_data}_{pop_year}'
    title_pop_string = f'{pop_data} {pop_year}'

# Water Depths to Compute damages, based on Mean, Maximum, and/or Minimum
# depths at a building's location ['Mean', 'Max', 'Min'],
# - mean = mean water depth at a building's location
# - max = maximum mean water depth at a building's location
# - min = minimum water depth at a building's
depth_stats = ['mean', 'max', 'min']
depth_stat = 'mean'
if depth_stat not in depth_stats:
    raise ValueError(f'ERROR: {depth_stat} must be one of: {depth_stats}')

# Directory of main folder
os.makedirs(dir_work, exist_ok=True)
os.chdir(dir_work)

# Name of directory to save files for flood risk assessment
dir_flood_cra = os.path.join(dir_work, file_prefix)
os.makedirs(dir_flood_cra, exist_ok=True)

# A user defined bounding box or GeoJSON or Shapefile can be used to define the
# flood region of interest.
#  Option 1 (if_bounding_file = True): GeoJSON or Shapefile
#   - The GeoJSON file (or Shapefile) can be generated at https://geojson.io.
#   - The file should be uploaded into the dir_flood_cra folder
#   - The spatial reference system should be EPSG:4326
#  Option 2 (if_bounding_file = False): Define latitude and longitude bounding box
#   - Define the maximum and minimum latitudes and longitudes
if_bounding_file = True
if if_bounding_file:  # Only if no GeoJSON or Shapefile is provided
  path_vect_bbox = f'{dir_work}/{file_prefix}.geojson'
  # Check if region of interest file exists
  if not os.path.exists(path_vect_bbox):
    raise FileNotFoundError(f'geojson or shapefile does not exist: {path_vect_bbox}')
else:  # Only define if GeoJSON or Shapefile is provided
  lat1, lat2 = 43.76, 43.77
  lon1, lon2 = 11.25, 11.28


## Water Depths for Exposed and Displaced Population---------------
# If depth is greater than this, the population is exposed
depth_min_exposed = 0.01
# If depth is greater than this, the population is displaced
depth_min_displaced = 1.0


## Figure Options ------------------------------------------
# Save images in folder (True saves)
if_save_fig = True

# Return period for the optional figures.
# - EU values: [10, 20, 30, 40, 50, 75, 100, 200, 500]
# - Global values: [10, 20, 50, 75, 100, 200, 500]
# At least one input, MUST also be present in return_periods
return_periods_fig = [10, 50, 100, 500]

# Buffer size, i.e., additional space around the region of interest in the maps
ybuffer=0.0020
xbuffer=0.0040

# Outline of location settings
if_show_outline = True #Set to True to show outline, False to hide it
outline_color = 'limegreen' #Colour of outline
outline_style = '-' #Style of the outline. Choose from -, :, --, -.
outline_thickness = 3 #Line thickness of outline
outline_face = 'none' #Face colour of outline
outline_alpha = 1 #Transparency of edge and face colour for the outline
if_outline_label = False #True prints the outline label, False hides it
outline_label = f'{file_prefix} Outline' #Text used for the label of the outline
outline_labelSize = 8 #Fontsize of otuline label
outline_labelPosition = 'upper left' #Label position, e.g., 'upper left', 'lower right'
legend_outline = None

# Maximum Water level in legend
# -  Useful if big discrepancy between river and flood depths exists.
# customMaxDepthLegend = -1 set highest data value as legend's maximum
customMaxDepthLegend = -1

# Damage curves (True prints)
if_damage_curve = True

# Building Images (True prints)
# Building maps with classification
if_building = True
# Building maps with flood level
if_building_h2o = True
# Building maps with damage received
if_building_dmg = True
# Annual damage by return period graph
if_building_dmg_plot = True

# Population Images (True prints)
    #Population exposed map
If_population_exp = True
    #Annual population exposed graph
If_population_exp_plot = True
    #Population displaced map
if_pop_displaced = True
    #Annual population displaced graph
if_pop_displaced_plot = True

## Figure Colour Scheme -------------------------------
# Water Depth Colorbar for Maps
# - cmap_h2o='Blues' can also be a good option
#cmap_h2o = LinearSegmentedColormap.from_list('gist_stern_inv', ['blue', 'red'],
#                                             N=256)
cmap_h2o = 'YlGnBu'
cmap_h2o = 'Blues'
# Building Class Colorbar for Maps
cmap_cls = LinearSegmentedColormap.from_list('gist_stern_inv',
                                             ['orange','purple', 'blue', 'red'],
                                             N=256)
# Population Colorbar for Maps
cmap_pop = LinearSegmentedColormap.from_list('gist_stern_inv',
                                             ['orange', 'red','fuchsia'],
                                             N=256)
cmap_pop = 'YlOrRd'
# Damage Colorbar for Maps
cmap_dmg = LinearSegmentedColormap.from_list('gist_stern_inv',
                                             ['blue', 'red','fuchsia'],
                                             N=256)
cmap_dmg = 'gist_stern_r'
## Data Management ----------------------------------
# - Temporary initiation. Changes in workflow
RP, tile_id_max = True, 0


# Downloads Folder
dir_downloads = os.path.join(dir_work, 'Downloads')
os.makedirs(dir_downloads, exist_ok=True)

# Downloaded Flood Depth Data
dir_flood_data_dl = os.path.join(dir_downloads, 'Flood')
os.makedirs(dir_flood_data_dl, exist_ok=True)

# Downloaded Population Data
dir_pop_data_dl = os.path.join(dir_downloads, 'Population')
os.makedirs(dir_pop_data_dl, exist_ok=True)

# Downloaded Landuse Data
dir_lu_data_dl = os.path.join(dir_downloads, 'Landuse')
os.makedirs(dir_lu_data_dl, exist_ok=True)

# OSM Path
dir_osm_data = os.path.join(dir_flood_cra, 'OSM')
os.makedirs(dir_osm_data, exist_ok=True)

# Flood Depths for AOI directory
dir_flood = os.path.join(dir_flood_cra, 'Flood')
os.makedirs(dir_flood, exist_ok=True)

# Damage results directory
dir_damage = os.path.join(dir_flood_cra, 'Damage')
os.makedirs(dir_damage, exist_ok=True)

# Population results directory
dir_pop = os.path.join(dir_flood_cra, 'Population')
os.makedirs(dir_pop, exist_ok=True)

# Landuse results directory
dir_lu = os.path.join(dir_flood_cra, 'Landuse')
os.makedirs(dir_lu, exist_ok=True)

# Saved figures directory
dir_figures = os.path.join(dir_flood_cra, 'Figures')
os.makedirs(dir_figures, exist_ok=True)

print('Done with input definitions')

In [ ]:
if flood_data == 'WRI':
    # Authenticate and initialize GEE
    import geemap
    import ee
    #   If EE gives problems run the below command and follow browser steps
    #!earthengine authenticate --force
    ee.Authenticate()
    ee.Initialize()

# Check Region of Interest

In [ ]:
def get_spatial_bounds(file_path):
    """
    Reads GeoJSON or Shapefile and extracts the min/max latitude and longitude.

    Args:
        file_path (str): Path to the spatial file (.geojson or .shp).

    Returns:
        tuple: (min_latitude, max_latitude, min_longitude, max_longitude)
    """
    # Load the file into a GeoDataFrame
    gdf = gpd.read_file(file_path)
    if gdf.crs is None:
      raise ValueError('File does not have a CRS defined')
    if gdf.crs.to_string() != 'EPSG:4326':
      print(f'EPSG:4326 is required, but file uses {gdf.crs} instead')
      raise ValueError('Incorrect spatial reference system')
    outline_geometry=gdf.geometry
    if outline_geometry.ndim>1:
        raise ValueError(f'Make sure file contains only the needed outline'
        '(polygon) so that the variable dimension is 1. Current dimensions: '
        '{outline_geometry.ndim}')
    # min_lon, min_lat, max_lon, max_lat since EPSG:4326
    minx, miny, maxx, maxy = gdf.total_bounds

    return outline_geometry, miny, maxy, minx, maxx


if if_bounding_file:
  outline_geometry, lat1, lat2, lon1, lon2 = get_spatial_bounds(path_vect_bbox)
else:
  lat1, lat2 = min(lat1, lat2), max(lat1, lat2)
  lon1, lon2 = min(lon1, lon2), max(lon1, lon2)
  outline_geometry=gpd.GeoSeries(
      [Polygon([(lon1, lat1), (lon2, lat1),
       (lon2, lat2), (lon1, lat2)])],crs='EPSG:4326')
  path_vect_bbox = os.path.join(dir_flood_cra, f'{file_prefix}_bbox.geojson')
  # Define the bounding box coordinates based on the zoomed region
  bounding_box = gpd.GeoDataFrame(geometry=outline_geometry)
  # Write the GeoDataFrame to a shapefile
  bounding_box.to_file(path_vect_bbox)
  print('Bounding box vector file written to: '+path_vect_bbox)
print(f'Bounding Box: ({lat1}, {lat2}, {lon1}, {lon2})')

fig, ax = plt.subplots(figsize=(10, 10))
plt.title('Check if Region of Interest is Mapped Correctly')
outline_geometry.plot(ax=ax, edgecolor='darkred', facecolor='none', linewidth=3)
ctx.add_basemap(ax, crs=gpd.read_file(path_vect_bbox).crs.to_string(),
                source=ctx.providers.OpenStreetMap.Mapnik,
                attribution=None, alpha=1)
plt.show()

## Define depth-damage functions
Damage caused to buildings can be determined in relation to flood depth that the buildings are subjected to. In this section the relationship between water depth and damage are determined.

Maximum damage values are based on:
- Huizinga, J., Moel, H. de, Szewczyk, W. (2017). Global flood depth-damage functions. Methodology and the database with guidelines. EUR 28552 EN. doi: 10.2760/16510
With following assumed values:
- CPI2010 = 2010 [World Bank Consumer Price Index](https://data.worldbank.org/indicator/FP.CPI.TOTL) for country of interest.
- CPI2022 = 2022 Consumer Price Index for country of interest (latest value).
- In calculating maximum damage, first array value is 2010 building reconstruction costs per square meter.
- Second array value is 2010 building content replacement value per square meter.

Damage classes:
- Options are Residential, Commercial, Industrial, Agriculture, Cultural, and Transportation, as well as an Universal class.
- In the default code, Agriculture, Cultural, and Transportation as well as unclassified buildings are set to Universal.

Damage function:
- Based on polynomial functions fit to the JRC depth-damage curves, with the order depending on fit (coefs).
- In the default code, a combined damage function is applied based on Residential, Commercial, and Industrial JRC depth-damage values

In [ ]:
# Define arrays for damage values based on 2010 estimates
cpi2010 = 100                                  # 2010 EU Consumer Price Index Value
cpi2022 = 121.8                                # 2022 EU Consumer Price Index Value
cpi_frac = cpi2022 / cpi2010
# 1st value = Recontruction costs; 2nd=contents
maxdmg_res = np.array([480, 240]) * cpi_frac    # EU Value, Residential
maxdmg_com = np.array([502, 502]) * cpi_frac    # EU Value, Commercial
maxdmg_ind = np.array([328, 492]) * cpi_frac    # EU Value, Industrial
maxdmg_agr = np.array([0.23, 0.46]) * cpi_frac  # Italy 2021 (AGR), Agricultural, currently not used
maxdmg_cul = maxdmg_com                          # EU Value, Cultural, currently not used
maxdmg_trs = maxdmg_ind                          # Italy 2021 (TRS), Transport, currently not used
maxdmg_uni = (maxdmg_res+maxdmg_com+maxdmg_ind)/3  # Universal class
# Combine damage arrays into a single array
maxdmg = np.column_stack((maxdmg_res, maxdmg_com, maxdmg_ind, maxdmg_uni))

# Damage classes
dmg_classes = ['Residential', 'Commercial', 'Industrial' ,'Universal', 'Agricultural']

def damage_function(wd1, coefs, wd_range=(0, 6)):
    wd = np.clip(wd1, *wd_range)
    y = coefs[0] * wd**5 + coefs[1] * wd**4 + coefs[2] * wd**3 \
        + coefs[3] * wd**2 + coefs[4] * wd + coefs[5]
    y = np.clip(y, 0, 1)
    return y

# Polynomial coefficients for each function
#   - Up to 5th order
#   - 1st value is highest order (5th) and last is intercept
coefs_uni = [0.0,  0.0, 0.0, -0.02787, 0.3334, 0.0]
coefs_res = [0.0005869, -0.01077, 0.07497, -0.2602, 0.5959, 0.0]
coefs_com = [0.0, 0.0, -0.0009149, -0.02021, 0.3216, 0.0]
coefs_ind = [0.0, 0.0, -0.001202, -0.01225, 0.2852, 0.0]
coefs_trs = [0.0, -0.00938, 0.07734, -0.2906, 0.7625, 0.0]
coefs_agr = [0.0, -0.004601, 0.06114, -0.3061, 0.7773, 0.0]

# Plot Depth Damage Functions
# Water depth values from 0 to 6m
wd_values = np.linspace(0, 6, 100)
dmg_res = damage_function(wd_values, coefs_res, wd_range=(0, 6))
dmg_com = damage_function(wd_values, coefs_com, wd_range=(0, 5))
dmg_ind = damage_function(wd_values, coefs_ind, wd_range=(0, 5))
dmg_uni = damage_function(wd_values, coefs_uni, wd_range=(0, 6))
dmg_agr = damage_function(wd_values, coefs_agr, wd_range=(0, 5))
if if_damage_curve:
    plt.plot(wd_values, dmg_uni, color='black', linewidth=3, label='Universal')
    plt.plot(wd_values, dmg_res, color='blue', linestyle='--', linewidth=1.5,
            label='Residential')
    plt.plot(wd_values, dmg_com, color='orange', linestyle='--', linewidth=1.5,
            label='Commercial')
    plt.plot(wd_values, dmg_ind, color='darkred', linestyle='--', linewidth=1.5,
            label='Industrial')
    plt.plot(wd_values, dmg_agr, color='darkgreen', linestyle='--', linewidth=1.5,
            label='Agricultural')
    plt.grid(color='grey', linestyle='-', linewidth=0.5)
    plt.xlabel('Water Depth (m)')
    plt.ylabel('Damage Fraction')
    plt.title('JRC Depth-Damage Functions')
    plt.legend()
    plt.show()

print('Done preparing detph-damage curves')

# Mapping Functions

In [ ]:
def map_vector(
    gdf,
    column,
    outline_geometry,
    lat1,
    lat2,
    lon1,
    lon2,
    cmap,
    vmin,
    vmax,
    title='Map of Vector Data',
    legend_label='Legend',
    if_classes=False,
    outline_color='black',
    outline_style='-',
    outline_thickness=1,
    outline_face='none',
    outline_alpha=1.0,
    outline_label='Outline',
    outline_labelPosition='upper right',
    outline_labelSize='small',
    if_show_outline=True,
    if_outline_label=True,
    xbuffer=0,
    ybuffer=0,
    if_save_fig=True,
    path_fig='vect.png'):

    fig, ax = plt.subplots(figsize=(8, 8))
    plt.title(title)
    plt.text(0.02, 0.02, f'Projection in {gdf.crs}', fontsize=8,
             path_effects=[pe.withStroke(linewidth=3, foreground='white')],
             transform=ax.transAxes, zorder=7)

    if if_show_outline:
        outline_geometry.plot(ax=ax, edgecolor=outline_color, linestyle=outline_style,
                             linewidth=outline_thickness, facecolor=outline_face,
                             alpha=outline_alpha, zorder=5)

    if if_outline_label:
        outline_legend = mlines.Line2D([], [], color=outline_color,
                                       linewidth=outline_thickness, label=outline_label)
        legend_outline = ax.legend(handles=[outline_legend], loc=outline_labelPosition,
                                   fontsize=outline_labelSize, frameon=True)

    if if_classes:
        gdf.plot(column=column, ax=ax, legend=True, cmap=cmap,
                 legend_kwds={'ncol': 4, 'bbox_to_anchor': (0.5, -0.15),
                              'loc': 'upper center', 'title': legend_label})
    else:
        # Plot the main data and capture the Axes
        gdf.plot(column=column, vmin=vmin, vmax=vmax, cmap=cmap, ax=ax)
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="2%", pad=0.01)
        main_collection = ax.collections[-1]
        plt.colorbar(main_collection, cax=cax, label=legend_label, extend='max')

    ax.set_xlim(lon1 - xbuffer, lon2 + xbuffer)
    ax.set_ylim(lat1 - ybuffer, lat2 + ybuffer)
    ax.axis('off')

    ctx.add_basemap(ax=ax, crs=gdf.crs, source=ctx.providers.OpenStreetMap.Mapnik,
                    attribution=None, alpha=0.5)
    if ax.texts:  # Check if any text objects are present
        for txt in ax.texts:
            txt.set_visible(False)  # Make them invisible or use `txt.remove()` to delete
    txt = ax.texts[-1]
    txt.set_position([0.99, 0.98])
    txt.set_ha('right')
    txt.set_va('top')

    if if_outline_label:
        ax.add_artist(legend_outline)

    if if_save_fig:
        plt.savefig(path_fig, bbox_inches='tight')
    plt.show()


def map_raster(
    rast_data,
    title,
    legend_label,
    vmin,
    vmax,
    cmap,
    xMin,
    xMax,
    yMin,
    yMax,
    lat_origin,
    epsg_rast,
    lon1,
    lon2,
    lat1,
    lat2,
    xbuffer,
    ybuffer,
    if_show_outline,
    outline_geometry,
    outline_color='black',
    outline_style='-',
    outline_thickness=1,
    outline_face='none',
    outline_alpha=1.0,
    outline_label='Outline',
    outline_labelPosition='upper right',
    outline_labelSize='small',
    if_outline_label=True,
    text_label=None,
    x_label=0.0,
    y_label=0.0,
    alpha=0.7,
    if_save_fig=True,
    path_fig='rast.png'
):
    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(rast_data, vmin=vmin, vmax=vmax, cmap=cmap,
                   extent=(xMin, xMax, yMin, yMax),
                   zorder=2, alpha=alpha, origin=lat_origin)
    plt.title(title)
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='2%', pad=0.1)
    plt.colorbar(im, cax=cax, label=legend_label, extend='max')
    plt.text(0.02, 0.02, f'Projection in {epsg_rast}', fontsize=8,
             path_effects=[pe.withStroke(linewidth=3, foreground='white')],
             transform=ax.transAxes, zorder=7)
    ax.set_xlim(lon1 - xbuffer, lon2 + xbuffer)
    ax.set_ylim(lat1 - ybuffer, lat2 + ybuffer)
    ctx.add_basemap(ax=ax, crs=epsg_rast,
                    source=ctx.providers.OpenStreetMap.Mapnik,
                    attribution=None, alpha=0.4)
    # Remove OSM text
    if ax.texts:
        for txt in ax.texts:
            txt.set_visible(False)
    txt = ax.texts[-1]
    txt.set_position([0.99, 0.98])
    txt.set_ha('right')
    txt.set_va('top')
    if text_label is not None:
        plt.text(x_label, y_label, text_label, transform=ax.transAxes,
                 fontsize=8, verticalalignment='top', horizontalalignment='left',
                 bbox=dict(facecolor='white', alpha=0.5, edgecolor='black'))
    if if_show_outline:
        outline_geometry.plot(ax=ax, edgecolor=outline_color,
                             linestyle=outline_style,
                             linewidth=outline_thickness,
                             facecolor=outline_face,
                             alpha=outline_alpha, zorder=5)
    if if_outline_label:
        outline_legend = mlines.Line2D([], [], color=outline_color,
                                       linewidth=outline_thickness,
                                       label=outline_label)
        legend_outline = ax.legend(handles=[outline_legend],
                                   loc=outline_labelPosition,
                                   fontsize=outline_labelSize, frameon=True)

    ax.axis('off')

    if if_save_fig:
        plt.savefig(path_fig, bbox_inches='tight')
    plt.show()

print('Mapping functions loaded')

def cbar_magnitude(number):
    magnitude = 10 ** (int(np.log10(number)))
    return round(number / magnitude) * magnitude

## Download data
In this section, the data required to run the analysis is downloaded.
- Download flood depth and population rasters to the data folder if the file doesn't exist.
- River flood extent and water depth are from the [Copernicus Land Monitoring Service](https://data.jrc.ec.europa.eu/dataset/1d128b6c-a4ee-4858-9e34-6210707f3c81) for different return periods, with 3 arc-seconds resolution (30-75m in Europe).
- Population densities are from the [European Commission's Joint Research Centre](https://data.jrc.ec.europa.eu/dataset/2ff68a52-5b5b-4a22-8f40-c41da8332cfe), with 3 arc-seconds resolution (30-75m in Europe).
- Load landuse data based on [GCAM-Demeter-LU](https://data.pnnl.gov/group/nodes/dataset/13192) projections, filtered for crops

This section can be modified to use local data.

In [ ]:
# Functions for downloading flood and population rasters

def download_files(file_urls, dir_data, max_retries=3, timeout=90):
    """Download files"""
    downloaded_files = []

    os.makedirs(dir_data, exist_ok=True)

    for file_url in file_urls:

        file_local = os.path.join(dir_data,os.path.basename(file_url))
        downloaded_files.append(file_local)

        # Download with retries
        if not os.path.exists(file_local):
            for attempt in range(max_retries):
                try:
                    response = requests.get(file_url, stream=True,
                                            timeout=timeout)
                    response.raise_for_status()

                    with open(file_local, 'wb') as f, tqdm(
                        desc=f'  Downloading {os.path.basename(file_url)}',
                        total=int(response.headers.get('content-length', 0)),
                        unit='B',
                        unit_scale=True,
                        unit_divisor=1024,
                    ) as bar:
                        for data in response.iter_content(chunk_size=1024):
                            f.write(data)
                            bar.update(len(data))

                    break
                except Exception as e:
                    if attempt == max_retries - 1:
                        print(f'Failed to download {file_url} after {max_retries} attempts')
                        raise
                    continue
        else:
            print(f'File exists. Skipping: {file_local}')

    return downloaded_files

def global_flood_tiles(lat1, lat2, lon1, lon2):
    """Determine 10x10° tiles needed for a given region"""
    file_prefixes = [
        'ID1_N80_W170',   'ID2_N70_W170',   'ID3_N60_W170',   'ID4_N80_W160',
        'ID5_N70_W160',   'ID6_N60_W160',   'ID7_N80_W150',   'ID8_N70_W150',
        'ID9_N60_W150',   'ID10_N80_W140',  'ID11_N70_W140',  'ID12_N60_W140',
        'ID13_N80_W130',  'ID14_N70_W130',  'ID15_N60_W130',  'ID16_N50_W130',
        'ID17_N40_W130',  'ID18_N80_W120',  'ID19_N70_W120',  'ID20_N60_W120',
        'ID21_N50_W120',  'ID22_N40_W120',  'ID23_N30_W120',  'ID24_N80_W110',
        'ID25_N70_W110',  'ID26_N60_W110',  'ID27_N50_W110',  'ID28_N40_W110',
        'ID29_N30_W110',  'ID30_N20_W110',  'ID32_N80_W100',  'ID33_N70_W100',
        'ID34_N60_W100',  'ID35_N50_W100',  'ID36_N40_W100',  'ID37_N30_W100',
        'ID38_N20_W100',  'ID40_N80_W90',   'ID41_N70_W90',   'ID42_N60_W90',
        'ID43_N50_W90',   'ID44_N40_W90',   'ID45_N30_W90',   'ID46_N20_W90',
        'ID47_N10_W90',   'ID48_N0_W90',    'ID50_N80_W80',   'ID51_N70_W80',
        'ID52_N60_W80',   'ID53_N50_W80',   'ID54_N40_W80',   'ID55_N30_W80',
        'ID56_N20_W80',   'ID57_N10_W80',   'ID58_N0_W80',    'ID59_S10_W80',
        'ID60_S20_W80',   'ID61_S30_W80',   'ID62_S40_W80',   'ID63_S50_W80',
        'ID65_N80_W70',   'ID66_N70_W70',   'ID67_N60_W70',   'ID68_N50_W70',
        'ID69_N20_W70',   'ID70_N10_W70',   'ID71_N0_W70',    'ID72_S10_W70',
        'ID73_S20_W70',   'ID74_S30_W70',   'ID75_S40_W70',   'ID76_S50_W70',
        'ID77_N80_W60',   'ID78_N70_W60',   'ID79_N60_W60',   'ID80_N50_W60',
        'ID81_N10_W60',   'ID82_N0_W60',    'ID83_S10_W60',   'ID84_S20_W60',
        'ID85_S30_W60',   'ID87_N70_W50',   'ID88_N60_W50',   'ID89_N10_W50',
        'ID90_N0_W50',    'ID91_S10_W50',   'ID92_S20_W50',   'ID94_N70_W40',
        'ID95_N0_W40',    'ID96_S10_W40',   'ID97_N80_W30',   'ID98_N70_W30',
        'ID99_N80_W20',   'ID100_N70_W20',  'ID101_N60_W20',  'ID102_N30_W20',
        'ID103_N20_W20',  'ID104_N10_W20',  'ID105_N60_W10',  'ID106_N50_W10',
        'ID107_N40_W10',  'ID108_N30_W10',  'ID109_N20_W10',  'ID110_N10_W10',
        'ID111_N70_W0',   'ID112_N60_W0',   'ID113_N50_W0',   'ID114_N40_W0',
        'ID115_N30_W0',   'ID116_N20_W0',   'ID117_N10_W0',   'ID118_N0_W0',
        'ID119_N70_E10',  'ID120_N60_E10',  'ID121_N50_E10',  'ID122_N40_E10',
        'ID123_N30_E10',  'ID124_N20_E10',  'ID125_N10_E10',  'ID126_N0_E10',
        'ID127_S10_E10',  'ID128_S20_E10',  'ID129_S30_E10',  'ID130_N80_E20',
        'ID131_N70_E20',  'ID132_N60_E20',  'ID133_N50_E20',  'ID134_N40_E20',
        'ID135_N30_E20',  'ID136_N20_E20',  'ID137_N10_E20',  'ID138_N0_E20',
        'ID139_S10_E20',  'ID140_S20_E20',  'ID141_S30_E20',  'ID142_N80_E30',
        'ID143_N70_E30',  'ID144_N60_E30',  'ID145_N50_E30',  'ID146_N40_E30',
        'ID147_N30_E30',  'ID148_N20_E30',  'ID149_N10_E30',  'ID150_N0_E30',
        'ID151_S10_E30',  'ID152_S20_E30',  'ID153_S30_E30',  'ID154_N70_E40',
        'ID155_N60_E40',  'ID156_N50_E40',  'ID157_N40_E40',  'ID158_N30_E40',
        'ID159_N20_E40',  'ID160_N10_E40',  'ID161_N0_E40',   'ID162_S10_E40',
        'ID163_S20_E40',  'ID164_N70_E50',  'ID165_N60_E50',  'ID166_N50_E50',
        'ID167_N40_E50',  'ID168_N30_E50',  'ID169_N20_E50',  'ID170_N10_E50',
        'ID171_S10_E50',  'ID172_N80_E60',  'ID173_N70_E60',  'ID174_N60_E60',
        'ID175_N50_E60',  'ID176_N40_E60',  'ID177_N30_E60',  'ID178_N80_E70',
        'ID179_N70_E70',  'ID180_N60_E70',  'ID181_N50_E70',  'ID182_N40_E70',
        'ID183_N30_E70',  'ID184_N20_E70',  'ID185_N10_E70',  'ID186_N80_E80',
        'ID187_N70_E80',  'ID188_N60_E80',  'ID189_N50_E80',  'ID190_N40_E80',
        'ID191_N30_E80',  'ID192_N20_E80',  'ID193_N10_E80',  'ID194_N80_E90',
        'ID195_N70_E90',  'ID196_N60_E90',  'ID197_N50_E90',  'ID198_N40_E90',
        'ID199_N30_E90',  'ID200_N20_E90',  'ID201_N10_E90',  'ID202_N0_E90',
        'ID203_N80_E100', 'ID204_N70_E100', 'ID205_N60_E100', 'ID206_N50_E100',
        'ID207_N40_E100', 'ID208_N30_E100', 'ID209_N20_E100', 'ID210_N10_E100',
        'ID211_N0_E100',  'ID212_N80_E110', 'ID213_N70_E110', 'ID214_N60_E110',
        'ID215_N50_E110', 'ID216_N40_E110', 'ID217_N30_E110', 'ID218_N20_E110',
        'ID219_N10_E110', 'ID220_N0_E110',  'ID221_S10_E110', 'ID222_S20_E110',
        'ID223_S30_E110', 'ID224_N80_E120', 'ID225_N70_E120', 'ID226_N60_E120',
        'ID227_N50_E120', 'ID228_N40_E120', 'ID229_N30_E120', 'ID230_N20_E120',
        'ID231_N10_E120', 'ID232_N0_E120',  'ID233_S10_E120', 'ID234_S20_E120',
        'ID235_S30_E120', 'ID236_N80_E130', 'ID237_N70_E130', 'ID268_N60_E170',
        'ID269_S30_E170', 'ID270_S40_E170',
    ]

    # Calculate southern boundaries (latitude)
    southern_start = math.floor(lat1 / 10) * 10 + 10
    southern_end = math.floor((lat2 - 1e-9) / 10) * 10 + 10  # Handle edge cases
    southern_bounds = list(range(southern_start, southern_end + 10, 10))

    # Calculate western boundaries (longitude)
    western_start = math.floor(lon1 / 10) * 10
    western_end = math.floor((lon2 - 1e-9) / 10) * 10
    western_bounds = list(range(western_start, western_end + 10, 10))
    # Generate all possible tile combinations
    flood_tiles = []
    for slat in southern_bounds:
        for wlon in western_bounds:
            # Format latitude direction
            lat_ns = 'N' if slat >= 0 else 'S'
            lat_str = f"{lat_ns}{abs(slat)}"

            # Format longitude direction
            lon_ew = 'E' if wlon > 0 else 'W'
            lon_str = f"{lon_ew}{abs(wlon)}"

            flood_tile_string = f'{lat_str}_{lon_str}'
            file_prefix = next((s for s in file_prefixes if flood_tile_string in s), None)

            flood_tiles.append(file_prefix)

    return flood_tiles

def merge_global_flood_tiles(flood_raster_tiles, file_raster_merge):
    """
    Merge flood tiles into a single raster and save the output.
    """

    # Open the source files
    src_files = [rasterio.open(f) for f in flood_raster_tiles]

    # Merge the source files
    mosaic, transform = merge(src_files)

    # Update metadata
    meta = src_files[0].meta.copy()
    meta.update({
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": transform
    })

    # Save the merged raster
    with rasterio.open(file_raster_merge, 'w', **meta) as dst:
        dst.write(mosaic)

    # Print a message and append the output file path
    print(f'    Merged raster: {file_raster_merge}')

    return

def get_population_tiles(lat1, lat2, lon1, lon2, dir_dl,
                         max_retries=3, timeout=90):
    """Determine required population tiles using shapefile intersection."""
    # Download shapefile if not exists
    url_zip = 'https://ghsl.jrc.ec.europa.eu/download/GHSL_data_4326_shapefile.zip'
    path_zip = os.path.join(dir_dl, os.path.basename(url_zip))
    path_vect = os.path.join(dir_dl, 'WGS84_tile_schema.shp')
    if not os.path.exists(path_vect):
        zip_path = os.path.join(dir_dl, 'GHSL_data_4326_shapefile.zip')
        if download_files([url_zip], dir_dl,
                          max_retries=max_retries, timeout=timeout):
            with ZipFile(path_zip, 'r') as zip_ref:
                zip_ref.extractall(dir_dl)

    # Create bounding box geometry
    gdf = gpd.read_file(path_vect)
    bbox = box(lon1, lat1, lon2, lat2)
    bbox_gdf = gpd.GeoDataFrame(geometry=[bbox], crs="EPSG:4326")

    # Find intersecting tiles
    intersecting_tiles = gpd.sjoin(gdf, bbox_gdf, predicate='intersects')
    return intersecting_tiles['tile_id'].tolist()

def download_population_data(pop_year, dir_dl, lat1, lat2, lon1, lon2,
                             max_retries=3, timeout=90):
    """Main function to download and merge population tiles."""
    tile_ids = get_population_tiles(lat1, lat2, lon1, lon2, dir_dl=dir_dl)
    print(f'Required population tiles: {tile_ids}')

    # Download all tiles
    base_url = f'https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GHSL/GHS_POP_GLOBE_R2023A/GHS_POP_E{pop_year}_GLOBE_R2023A_4326_3ss/V1-0/tiles/'
    raster_files = []

    for tile_id in tile_ids:
        zip_name = f'GHS_POP_E{pop_year}_GLOBE_R2023A_4326_3ss_V1_0_{tile_id}.zip'
        tif_name = zip_name.replace('.zip', '.tif')
        zip_path = os.path.join(dir_dl, zip_name)
        tif_path = os.path.join(dir_dl, tif_name)
        # Download ZIP if TIF doesn't exist
        if not os.path.exists(tif_path):
            urls = [f'{base_url}{zip_name}']
            print(urls)
            print(urls)
            print(urls)
            print(urls)
            print(urls)
            downloaded = download_files(urls, dir_dl,
                                       max_retries=max_retries,
                                       timeout=timeout)
            if downloaded:
                with ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extract(tif_name, dir_dl)
                os.remove(zip_path)
        raster_files.append(tif_path)

    # Merge tiles using existing flood merge function
    merged_path = os.path.join(dir_dl, f'POP_MERGED_{pop_year}.tif')
    if len(raster_files) > 1 and not os.path.exists(merged_path):
        merge_global_flood_tiles(raster_files, merged_path)
    else:
        merged_path = raster_files[0]

    return merged_path


print('Done loading fuctions for downloading raster data')

In [ ]:
# Flood Depth Rasters
# - Downloads European or global river data to identify potential flood
#   risk hotspots.
# - While river flood data are used in this demo, pluvial and coastal flood
#   depths can be added or subsituted.
# - Local data are recommended for a location specific and more accurate
#   analysis.
# - The standard projection used is 'EPSG:4326'
# - Note: There is a known issue with Mollweide projections.
#
# Population Rasters
# - Download population tiles for AOI and merge if necessary
# - Local data may be more accurate, but are often in vector format, which may
#   require signifant modifications to accomodate the data.

if flood_data == 'JRC-EU':
    return_periods = [10, 20, 30, 40, 50, 75, 100, 200, 500]
    url_eu_flood_data = 'https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/CEMS-EFAS/flood_hazard'
    raster_files = []
    url_rasters = []
    for rp in return_periods:
        file_rast = f'Europe_RP{rp}_filled_depth.tif'
        url_rasters.append(os.path.join(url_eu_flood_data, file_rast))
        path_rast = os.path.join(dir_flood_data_dl, file_rast)
        raster_files.append(path_rast)
        url_rast = os.path.join(url_eu_flood_data, file_rast)
        #depth_full_rasters = download_eu_flood_data(url_rast, path_rast,
        #                                            max_retries=5, timeout=90)
    depth_full_rasters = download_files(url_rasters, dir_flood_data_dl)
elif flood_data == 'JRC-Global':
    return_periods = [10, 20, 50, 75, 100, 200, 500]
    url_global_flood_data = 'https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/CEMS-GLOFAS/flood_hazard'
    # Required tiles
    flood_tile_prefixes = global_flood_tiles(lat1, lat2, lon1, lon2)
    print(f'Global Flood tiles prefixes: {flood_tile_prefixes}')
    # Download required tiles
    depth_full_rasters = []
    for rp in return_periods:
        flood_tile_urls = [f'{url_global_flood_data}/RP{rp}/{prefix}_RP{rp}_depth.tif' for prefix in flood_tile_prefixes]
        print(f'Processing tiles for RP={rp}-years')
        print([os.path.basename(f) for f in flood_tile_urls])
        flood_tile_rasters = download_files(flood_tile_urls, dir_flood_data_dl)
        # Merge tiles for each return period
        if len(flood_tile_prefixes) > 1:
            lat_strings = [s.split('_')[1] for s in flood_tile_prefixes]
            lon_strings = [s.split('_')[2] for s in flood_tile_prefixes]
            lat_string = f'{lat_strings[-1]}-{lat_strings[0]}' if len(lat_strings) > 1 else lat_strings[0]
            lon_string = f'{lon_strings[-1]}-{lon_strings[0]}' if len(lon_strings) > 1 else lon_strings[0]
            flood_raster_merge = f'{lat_string}_{lon_string}_RP{rp}_depth.tif'
            flood_raster_merge = os.path.join(dir_flood_data_dl, flood_raster_merge)
            print(flood_raster_merge)
            if not os.path.exists(flood_raster_merge):
                print('  Merging tiles (can take a couple of minutes)')
                merge_global_flood_tiles(flood_tile_rasters, flood_raster_merge)
            else:
                print(f'  Merged raster already exists (skip download): {flood_raster_merge}')
            depth_full_rasters.append(flood_raster_merge)
        else:
            depth_full_rasters.append(flood_tile_rasters[0])
    print(depth_full_rasters)
elif flood_data == 'WRI':
    flood_dataset = ee.ImageCollection('WRI/Aqueduct_Flood_Hazard_Maps/V2').select('inundation_depth')
    return_periods = [1, 2, 5, 10, 25, 50, 100, 250, 500, 1000]
    gee_bbox = ee.Geometry.Rectangle([lon1, lat1, lon2, lat2])
    depth_full_rasters = []

    for rp in return_periods:
        print(f'Processing: RP={rp}-years')
        wri_filters = [
            ee.Filter.eq('floodtype', f'inun{flood_type}'),
            ee.Filter.eq('returnperiod', rp),
            ee.Filter.eq('model', f'{model.zfill(14)}')
        ]
        tif_file_strs = [f'WRI_RP{rp}', flood_type]

        if flood_type == 'coast':
            wri_filters.extend([
                ee.Filter.eq('projection', slr_scenario),
                ee.Filter.eq('subsidence', subsidence)
            ])
            tif_file_strs.extend([subsidence, f'SLR{slr_scenario}'])

        tif_file_strs.append(model)
        if model != 'WATCH':
            wri_filters.append(ee.Filter.eq('year', year_scen))
            tif_file_strs.append(year_scen)

        # Filter and process the image collection
        flood_gee = flood_dataset.filter(ee.Filter.And(*wri_filters)).map(lambda img: img.clip(gee_bbox))

        if flood_gee.size().getInfo() == 0:
            print(f'No images found for RP={rp}. Using placeholder.')
            flood_gee = ee.ImageCollection([ee.Image.constant(0).rename("inundation_depth").clip(gee_bbox)])

        # Interpolate to finer grid
        if flood_rast_res < 1000:
            flood_gee = flood_gee.map(lambda img: img.resample('bilinear').clip(gee_bbox))
            flood_image = flood_gee.mosaic()
            tif_file_strs.append(f'{flood_rast_res}m')
            tif_filename = '_'.join(tif_file_strs) + '.tif'
            tif_path = os.path.join(dir_flood_data_dl, tif_filename)
            if not os.path.exists(tif_path):
                geemap.ee_export_image(
                    flood_image,
                    tif_path,
                    scale=flood_rast_res,
                    region=gee_bbox
                )
            else:
                print(f'  Flood raster already exists (skip download): {tif_path}')
        # Leave on native 1000m grid
        else:
            flood_image = ee.Image(flood_gee.first())
            native_scale = flood_image.projection().nominalScale().getInfo()
            native_crs = flood_image.projection().crs()
            tif_filename = '_'.join(tif_file_strs) + '.tif'
            tif_path = os.path.join(dir_flood_data_dl, tif_filename)
            if not os.path.exists(tif_path):
                geemap.ee_export_image(
                    ee_object=flood_image,
                    filename=tif_path,
                    scale=native_scale,
                    region=gee_bbox,
                    crs=native_crs,
                )
            else:
                print(f'  Flood raster already exists (skip download): {tif_path}')

        print(f'Data Written: {tif_path}')
        depth_full_rasters.append(tif_path)

else:
    raise ValueError('Invalid flood data option. '
    'Current choices are "JRC-EU", "JRC-Global", "WRI"')


# Population Data
if pop_data == 'GHSL':
    print('\nDownload GHSL Population Data')
    pop_full_raster = download_population_data(pop_year, dir_pop_data_dl,
                                               lat1, lat2, lon1, lon2)
elif pop_data == 'SEDAC':
    pop_full_raster = os.path.join(dir_pop_data_dl, f'{pop_ssp.lower()}_total_{pop_year}.nc4')
    if not os.path.exists(pop_full_raster):
        print(f'ERROR: {pop_data} POPULATION FILE DOES NOT EXIST: {pop_full_raster}')

print(f'\nPopulation File: {pop_full_raster}')


# Landuse Data
print('\nLanduse Data')
lu_full_nc = os.path.join(dir_lu_data_dl, f'GCAM_Demeter_LU_ssp{lu_ssp[3]}_rcp{lu_ssp[4:6]}_modelmean_{lu_year}_crop.nc')
if os.path.exists(lu_full_nc):
    print(f'Landuse NetCDF File: {lu_full_nc}')
else:
    print(f'ERROR: LANDUSE FILE DOES NOT EXIST: {lu_full_nc}')


print('\nDone downloading flood, population, and landuse gridded files')

### OpenStreetMap buildings data
In this section the OSM data are loaded based on the bounding box defined above. The extracted data represents the building use (unclassified) and is written to a shapefile.

In [ ]:
# Output shapefile for unclassified buildings
path_vect_osm = os.path.join(dir_osm_data, f'{file_prefix}_OSM_Building_Unclassified.geojson')

# Define tags for OSM data
tags = {'building': True,'amenity': True}
# Retrieve OSM geometries within the bounding box
print('Reading in OSM data')
gdf_osm = ox.features_from_polygon(outline_geometry.iloc[0], tags)
# Filter out non-Polygon geometries
gdf_osm = gdf_osm[gdf_osm.geom_type == 'Polygon']
# Confirm that all of the GDF elements are compatible with shp format
gdf_osm = gdf_osm.map(lambda x: str(x) if isinstance(x, list) else x)
# Clip the OSM data to the bounding box
gdf_osm = gpd.clip(gdf_osm, outline_geometry)
# Save the polygon-only gdp to shapefile
print(f'Saving OSM data to GeoJSON: {path_vect_osm}')
gdf_osm.to_file(path_vect_osm, driver='GeoJSON')
print('OSM data read in and saved to shapefile')
print('  '+path_vect_osm)


if if_building:
    # Plot data
    path_fig = os.path.join(dir_figures, f'{file_prefix}_OSMbuilding_preclassification.png')
    title='OSM Buildings: Prior to residential, commericial, industrial classification'
    legend_label = 'OSM Building Type'
    map_vector(gdf_osm, 'building', outline_geometry, lat1, lat2, lon1, lon2,
               'coolwarm', -99, -99, title=title, legend_label=legend_label,
               if_classes=True, outline_color=outline_color,
               outline_style=outline_style, outline_thickness=outline_thickness,
               outline_face=outline_face, outline_alpha=outline_alpha,
               outline_label='Area of Interest',
               outline_labelPosition=outline_labelPosition,
               outline_labelSize=outline_labelSize,
               if_show_outline=if_show_outline,
               if_outline_label=if_outline_label,
               xbuffer=xbuffer, ybuffer=ybuffer,
               if_save_fig=True, path_fig=path_fig)

print('Done downloading OSM data for AOI')

### Reproject OSM data to raster projection
In this section, in order to perform the damage analysis, the OSM and raster data need to be on the same Coordinate Reference System (CRS). The OSM are on an EPSG:4326 CRS and the raster data could be another.  In the cell, the OSM data are converted to match the raster CRS.

In [ ]:
# Determine Raster EPSG Code (only works with osgeo)
ds = gdal.Open(depth_full_rasters[0])
srs = osr.SpatialReference()
srs.ImportFromWkt(ds.GetProjection())
epsg_rast = f'EPSG:{srs.GetAuthorityCode(None)}'
ds = None

print(f'Water Depth Raster Projection: {epsg_rast}')

# Determine CRSs from shapefile and Raster
crs_osm = gdf_osm.crs

if crs_osm not in (epsg_rast, epsg_rast.lower()):
    print(f'Reprojecting from OSM {crs_osm} to raster EPSG:{epsg_rast}')
    # Read the GeoDataFrame if reprojection is needed
    gdf_osm = gpd.read_file(path_vect_osm)
    # Reproject the GeoDataFrame to match the raster CRS
    gdf_osm = gdf_osm.to_crs(epsg=epsg_rast)
    # Save the reprojected data back to the shapefile
    gdf_osm.to_file(path_vect_osm)
    print('  Overwriting reprojected data:', path_vect_osm)

else:
    print('  No reprojection performed. Both projections are the same:', crs_osm)


# Create a transformer for coordinate conversion
transformer = Transformer.from_crs('EPSG:4326', epsg_rast, always_xy=True)

# Convert bounding box coordinates to raster CRS
xMin, yMin = transformer.transform(lon1-xbuffer, lat1-ybuffer)
xMax, yMax = transformer.transform(lon2+xbuffer, lat2+ybuffer)

print('  Converted Coordinate Bounds with Buffer:')
print(f'    Longitudes: {lon1-xbuffer}E --> {xMin} meters & {lon2+xbuffer}E --> {xMax} meters')
print(f'    Latitudes: {lat1-ybuffer}N --> {yMin} meters & {lat2+ybuffer}N --> {yMax} meters')

print('Done')

## Retrieving data for the area of interest
In this section the bounding box is used to crop the data to the area of interest.
The code converts latitude and longitude values to the equivalent projection used by the flood map raster & writes bounding box to shapefile. We also define the bounding box for OpenStreetMap data.

If the region of interest is not as desired, the latitude and longitude values can be changed in the "Define inputs" section above.

In [ ]:
# Check that flood, population, and landuse files align with region of interest.

print(f'Water Depth Raster Projection: {epsg_rast}')

# Read GeoDataFrame from the bounding box shapefile
gdfBBox = gpd.read_file(path_vect_bbox)
crsRast = gdfBBox.crs

for rp in sorted(return_periods, reverse=True):
    depth_full_raster = next((s for s in depth_full_rasters if f'RP{rp}_' in s), None)
    # Read the raster using rasterio
    print(depth_full_raster)
    with rasterio.open(depth_full_raster) as src:
        window = from_bounds(xMin, yMin, xMax, yMax, src.transform)
        r_depths = src.read(1, window=window)
        r_depths = np.ma.masked_where((r_depths <= 0) | (r_depths > 1000), r_depths)
        max_depth  = r_depths.max()
        if customMaxDepthLegend == -1:
            maxDepthLegend = ((max_depth // 1) + 1)
        else:
            maxval=customMaxDepthLegend
        missing_data_value = src.nodata

    if rp == max(return_periods_fig):
        depth_max = cbar_magnitude(max_depth*0.75)
    if rp in return_periods_fig:
        path_fig = os.path.join(dir_figures, f'{file_prefix}_RP{rp}_depth.png')
        title = f'River Flood Depth: RP={rp}-year'
        legend_label = 'Depth (m)'
        lat_origin = 'upper'
        map_raster(r_depths, title, legend_label, 0, depth_max, cmap_h2o,
                   xMin, xMax, yMin, yMax, lat_origin, epsg_rast, lon1, lon2, lat1, lat2,
                   xbuffer, ybuffer, True, outline_geometry,
                   outline_color, outline_style, outline_thickness, outline_face,
                   outline_alpha, outline_label, outline_labelPosition,
                   outline_labelSize, if_outline_label, alpha=0.7,
                   if_save_fig=False, path_fig=path_fig)

print('Done preparing flood data for region of interest')

In [ ]:
if pop_data == 'GHSL':
    # Open population raster
    pop_raster = rio.open_rasterio(pop_full_raster).squeeze('band', drop=True)
    # Bounding box in WGS84 (EPSG:4326)
    bbox_4326 = box(lon1 - xbuffer, lat1 - ybuffer, lon2 + xbuffer, lat2 + ybuffer)
    # Create GeoDataFrame and reproject to raster's CRS
    gdf_bbox = gpd.GeoDataFrame(geometry=[bbox_4326], crs="EPSG:4326")
    gdf_bbox_proj = gdf_bbox.to_crs(pop_raster.rio.crs)
    # Clip raster using reprojected bounding box
    rPopulation = pop_raster.rio.clip(gdf_bbox_proj.geometry, all_touched=True)
    # Mask low population values (<0.1)
    rPopulation = rPopulation.where(rPopulation >= 0.1)
    # Get metadata
    missing_data_value = pop_raster.rio.nodata
    lat_origin = 'upper'
elif pop_data == 'SEDAC':
    # Open population NetCDF
    ds_sedac = xr.open_dataset(pop_full_raster)
    #if (ds_sedac.lon.min() >= 0) and (lon1 < 0):
    #    ds_sedac = ds_sedac.assign_coords(lon=(ds_sedac.lon % 360))
    # Clip using native coordinates (no reprojection needed)
    rPopulation = ds_sedac['Band1'].sel(lon=slice(lon1 - xbuffer, lon2 + xbuffer),
                                        lat=slice(lat1 - ybuffer, lat2 + ybuffer))
    # Mask low values and preserve metadata
    rPopulation = rPopulation.where(rPopulation >= 0.1)
    ds_sedac.close()
    lat_origin = 'lower'

max_population = rPopulation.max().item()
maxPopLegend = ((max_population // 100) + 1) * 100
path_fig = os.path.join(dir_figures, f'{file_prefix}_map_pop_{file_pop_string}.png')
title = f'Population: {title_pop_string}'
legend_label = 'Population'
map_raster(rPopulation, title, legend_label, 0, maxPopLegend, cmap_pop,
           lon1, lon2, lat1, lat2, lat_origin,
           epsg_rast, lon1, lon2, lat1, lat2, xbuffer, ybuffer,
           False, outline_geometry, outline_color, outline_style,
           outline_thickness, outline_face, outline_alpha, outline_label,
           outline_labelPosition, outline_labelSize, if_outline_label,
           alpha=0.7, if_save_fig=False, path_fig=path_fig)

print('Done preparing population data for region of interest')

In [ ]:
## Landuse map
# Open the raster file
ds = xr.open_dataset(lu_full_nc).sel(latitude=slice(lat2, lat1), longitude=slice(lon1, lon2))

xr_crop_rf = ds['crop_rf'].where(ds['crop_rf'] > 0)/100.
xr_crop_irr = ds['crop_irr'].where(ds['crop_irr'] > 0)/100.

path_fig = os.path.join(dir_figures, f'croprf_{lu_ssp}_{lu_year}.png')
title = f'Rainfed Crop Fraction: {lu_ssp} {lu_year}'
legend_label = 'Fraction'
map_raster(xr_crop_rf, title, legend_label, 0, 1, cmap_pop,
           lon1, lon2, lat1, lat2, 'upper',
           epsg_rast, lon1, lon2, lat1, lat2, xbuffer, ybuffer,
           True, outline_geometry, outline_color, outline_style,
           outline_thickness, outline_face, outline_alpha, outline_label,
           outline_labelPosition, outline_labelSize, if_outline_label,
           alpha=0.7, if_save_fig=False, path_fig=path_fig)

path_fig = os.path.join(dir_figures, f'cropirr_{lu_ssp}_{lu_year}.png')
title = f'Irrigated Crop Fraction: {lu_ssp}_{lu_year}'
legend_label = 'Fraction'
map_raster(xr_crop_irr, title, legend_label, 0, 1, cmap_pop,
           lon1, lon2, lat1, lat2, 'upper',
           epsg_rast, lon1, lon2, lat1, lat2, xbuffer, ybuffer,
           True, outline_geometry, outline_color, outline_style,
           outline_thickness, outline_face, outline_alpha, outline_label,
           outline_labelPosition, outline_labelSize, if_outline_label,
           alpha=0.7, if_save_fig=False, path_fig=path_fig)

lu_nc = f'{file_prefix}_{os.path.basename(lu_full_nc)}'
lu_nc = os.path.join(dir_lu, lu_nc)

ds.to_netcdf(lu_nc)

print('Done preparing landuse data for region of interest')

### Building classifications
In this section, the building types are classified to Residential, Commercial, Industrial, etc.
- This procedure needs to be performed manually by looking at the list of building types.
- If None is listed for a building class (bldgClass column), the building type (building column) should be assigned to one of the lists (e.g., class_res, class_com, class_ind).
- Buildings with type "yes" will be classified as Universal by default in a later step.

As calculated in an earlier section, the Universal class has a damage curve equal to the average of the Residential, Commercial, and Industrial ones.

Below we have added a classification of OSM building types already.

In [ ]:
# CSV file with classifications (class_res, class_com, etc)
csv_osm_classes = os.path.join(dir_osm_data, f'{file_prefix}_OSM_Building_Reclassified.csv')
csv_osm_amenity = os.path.join(dir_osm_data, f'{file_prefix}_OSM_Amenity_Classified.csv')
 # Keep only the building and geometry columns
gdf_buildings = gdf_osm[['building','amenity','geometry']].copy()

# Building classifications
#   - Change and add as needed
class_res = ['hut', 'apartments', 'detached', 'residential', 'house', 'barn',
             'garage', 'carport', 'semidetached_house', 'shed', 'bungalow',
             'roof', 'terrace', 'allotment_house', 'gazebo', 'shelter',
             'chimney']
class_com = ['commercial', 'office', 'retail', 'kiosk', 'supermarket',
             'warehouse', 'garages', 'hotel', 'stadium', 'grandstand',
             'sports_centre',  'pavilion', 'government', 'school',
             'kindergarten', 'university', 'dormitory', 'public', 'service',
             'hospital', 'civic', 'terminal', 'fire_station', 'train_station',
             'boathouse', 'toilets', 'tech_cab', 'tower', 'portal',
             'columbarium', 'greenhouse', 'guardhouse', 'construction',
             'funeral_hall', 'garden_folly','college', 'stable', 'services',
             'bunker', 'sports_hall', 'bleachers', 'container', 'silo',
             'farm_auxiliary', 'storage_tank', 'gasometer']
class_ind = ['industrial', 'manufacture', 'factory']
class_cul = ['church', 'cathedral', 'baptistery', 'obelisk', 'basilica',
             'monastery', 'ruins', 'column', 'chapel', 'synagogue', 'shrine',
             'religious', 'convent', 'fort', 'historic', 'bell_tower',
             'mosque', 'biblioteca']
class_agr = []
class_trs = ['bridge', 'parking', 'transportation', 'garbage_shed',
             'hangar', 'houseboat']
class_uni = ['universal', 'abandoned']

# Critical infrastructure defintions
# - Change and add as needed
# - Can classify critical infrastructure both through its building or amenity tag.
# - It is suggested to run this cell once, read the output below, and add the
#   building or amenities required in the critical infrastructure list.
list_crit_infrastructure = ['hospital','fuel','bank','clinic','pharmacy',
                            'police','prison','refugee_site', 'train_station',
                            'fire_station','transformer_tower','water_tower',
                            'bridge', 'transportation','school', 'post_office']
# This affects the look of the maps in the 'Critical Infrastructure' section
crit_marker_colors = {
    'hospital': {'marker': 'P', 'color': 'red'},
    'police': {'marker': 's', 'color': 'blue'},
    'train_station': {'marker': 'o', 'color': 'green'},
    #'bus_station': {'marker': 'o', 'color': 'green'},
    'transformer_tower': {'marker': '*', 'color': 'purple'},
    'water_tower': {'marker': 'v', 'color': 'orange'},
    'bridge': {'marker': 'd', 'color': 'brown'},
    'fire_station': {'marker': 'X', 'color': 'pink'},
    'transportation': {'marker': '>', 'color': 'cyan'},
    'refugee_site': {'marker': 'o', 'color': 'lime'},
    'fuel': {'marker': '2', 'color': 'yellow'},
    'post_office': {'marker': '2', 'color': 'red'},
    'school': {'marker': 'X', 'color': 'red'},
    #'kindergarten': {'marker': 'X', 'color': 'red'},
    #'college': {'marker': 'X', 'color': 'red'},
    #'university': {'marker': 'X', 'color': 'red'},
}

# For now, set transportation and cultural to universal (can add/change if desired)
class_uni = class_uni + class_cul + class_agr + class_trs
class_cul = []
class_agr = []
class_trs = []

# Convert building classes to dataframe
bldg_classes = pd.DataFrame({'building': gdf_buildings['building'].unique()})

# Classify each structure to Residential, Commercial, Industrial, etc.
#   - Structures not listed in above class lists are classified as none
bldg_classes['bldgClass'] = None
bldg_classes.loc[bldg_classes['building'].isin(class_res), 'bldgClass'] = 'Residential'
bldg_classes.loc[bldg_classes['building'].isin(class_com), 'bldgClass'] = 'Commercial'
bldg_classes.loc[bldg_classes['building'].isin(class_ind), 'bldgClass'] = 'Industrial'
bldg_classes.loc[bldg_classes['building'].isin(class_cul), 'bldgClass'] = 'Cultural'
bldg_classes.loc[bldg_classes['building'].isin(class_agr), 'bldgClass'] = 'Agricultural'
bldg_classes.loc[bldg_classes['building'].isin(class_trs), 'bldgClass'] = 'Transportation'
bldg_classes.loc[bldg_classes['building'].isin(class_uni), 'bldgClass'] = 'Universal'

#   - Adding critical infrastructure
bldg_classes['critInfrastructure'] = None
bldg_classes.loc[bldg_classes['building'].isin(list_crit_infrastructure),
                'critInfrastructure'] = True

amenity_classes = pd.DataFrame({'amenity': gdf_buildings['amenity'].unique()})
amenity_classes['critInfrastructure'] = None
amenity_classes.loc[amenity_classes['amenity'].isin(list_crit_infrastructure),
                   'critInfrastructure'] = True


# Write structure and classification to CSV
bldg_classes.to_csv(csv_osm_classes, index=False)
amenity_classes.to_csv(csv_osm_amenity, index=False)

print('Building Classifications')
print('- If "None" is listed for a building class, add the building one of the above lists.')
print('- Building "yes" will be classified as Universal in a later step.')
print(bldg_classes)
print(amenity_classes)

print('Done with reclassification definitions')

In the code below, the building types are classified and written to a GeoJSON with a column.
- The first map shows the buildings that have been classified based on the above assignments.
- The second map shows the same, but the with building type "yes" classified as Universal. The difference between the maps provides an idea of the how many building have a type assigned to them.

In [ ]:
# Write GeoJSON file with reclassfied buildings

# GeoJSON name with reclassfied buildings (output)
path_vect_osm_reclass = os.path.join(dir_osm_data,f'{file_prefix}_OSM_Building_Reclassified.geojson')

# Building classification created earlier
bldg_classes = pd.read_csv(csv_osm_classes)

# Merge the spatial data with the new information
gdf_osm_reclass = pd.merge(gdf_osm, bldg_classes, on='building', how='left')

if if_building:
    # Plot without unclassified buildings
    title = 'Building Data without Unclassified Buildings'
    legend_label = 'Building Classes'
    path_fig = os.path.join(dir_figures, f'{file_prefix}_OSMbuilding_unclassified_simple.png')
    map_vector(gdf_osm_reclass, 'bldgClass', outline_geometry,
               lat1, lat2, lon1, lon2, 'coolwarm', -99, -99, title=title,
               legend_label=legend_label, if_classes=True,
               outline_color=outline_color, outline_style=outline_style,
               outline_thickness=outline_thickness, outline_face=outline_face,
               outline_alpha=outline_alpha, outline_label='Area of Interest',
               outline_labelPosition=outline_labelPosition,
               outline_labelSize=outline_labelSize,
               if_show_outline=if_show_outline,
               if_outline_label=if_outline_label,
               xbuffer=xbuffer, ybuffer=ybuffer,
               if_save_fig=True, path_fig=path_fig)

# Substitute undefined (null) building classes
gdf_osm_reclass['bldgClass'] = gdf_osm_reclass['bldgClass'].fillna('Universal')

# Write classified structures to file
gdf_osm_reclass.to_file(path_vect_osm_reclass, encoding='utf-8')

if if_building:
    # Plot with unclassified buildings as classified
    title = 'Building Data with Unclassified Buildings as Universal Class'
    legend_label = 'Building Classes'
    path_fig = os.path.join(dir_figures, f'{file_prefix}_OSMbuilding_unclassified_simple.png')
    map_vector(gdf_osm_reclass, 'bldgClass', outline_geometry,
               lat1, lat2, lon1, lon2, 'coolwarm', -99, -99, title=title,
               legend_label=legend_label, if_classes=True,
               outline_color=outline_color, outline_style=outline_style,
               outline_thickness=outline_thickness, outline_face=outline_face,
               outline_alpha=outline_alpha, outline_label='Area of Interest',
               outline_labelPosition=outline_labelPosition,
               outline_labelSize=outline_labelSize,
               if_show_outline=if_show_outline,
               if_outline_label=if_outline_label,
               xbuffer=xbuffer, ybuffer=ybuffer,
               if_save_fig=True, path_fig=path_fig)

print('Done assigning classifications to buildings')

## Flood depths at building locations
In this section, the flood map rasters for each return period (extreme event) are loaded.
- The rasters are then translated to flood depths for each building based on the desired statistic (mean, maximum or minimum depth or all three).
- For each return period a plot of a flood map can be generated as well as a plot with the flood depths corresponding to each building.

In [ ]:
# Flood depths at buildings

depth_rasters = []
depth_vects = []
for rp in sorted(return_periods, reverse=True):

    depth_full_raster = next((s for s in depth_full_rasters if f'RP{rp}_' in s), None)
    depth_raster = os.path.join(dir_flood, f'{file_prefix}_{flood_data}_RP{rp}_depth.tif')
    depth_rasters.append(depth_raster)
    depth_vect = depth_raster.replace('.tif', '-damage.geojson')
    depth_vects.append(depth_vect)

    print(f'Computing Building Water Depths:  RP={rp}')
    print(f'  Loading depths: {depth_full_raster}')
    print(f'  Writing depths: {depth_raster}')

    # Keep only the building, bldgClass and geometry columns

    gdf_dmg = gdf_osm_reclass[['building', 'bldgClass', 'geometry']].copy()

    # Compute building areas in m2
    gdf_dmg_ESPG3035=gdf_dmg.to_crs(3035)
    gdf_dmg['Area_m2'] = gdf_dmg_ESPG3035.geometry.area

    # Read the raster using rasterio
    with rasterio.open(depth_full_raster) as src:
        r_depths, out_transform = mask(src, outline_geometry, crop=True)
        r_depths = r_depths[0]
        r_depths = np.ma.masked_where((r_depths <= 0) | (r_depths > 1000), r_depths)
        missing_data_value = src.nodata
        if missing_data_value is None:
            missing_data_value = -9999
        max_depth  = r_depths.max()
        with rasterio.open(
            depth_raster,
            'w',
            driver='GTiff',
            height=r_depths.shape[0],
            width=r_depths.shape[1],
            count=1,
            dtype=r_depths.dtype,
            crs=src.crs,
            transform=out_transform,
            nodata=missing_data_value
        ) as dst:
            dst.write(r_depths, 1)
    height, width = r_depths.shape
    xMinDepth, yMinDepth, xMaxDepth, yMaxDepth = array_bounds(height, width, out_transform)
    # Perform zonal statistics directly on the raster array
    result = rasterstats.zonal_stats(
        gdf_dmg,
        r_depths,
        nodata=src.nodata,
        affine=out_transform,
        stats=['mean', 'min', 'max'],
        all_touched=True
    )

    # Update geodataframe with zonal statistics
    stat_name = depth_stat.title() + 'Depth'
    gdf_dmg[stat_name] = [entry[depth_stat] for entry in result]

    if customMaxDepthLegend == -1:
        maxDepthLegend = ((max_depth // 1) + 1)
    else:
        maxDepthLegend=customMaxDepthLegend

    if rp in return_periods_fig and if_building_h2o:
        #map_raster(r_depths, title, legend_label, 0, maxDepthLegend, cmap_h2o,
        #          xMinDepth, xMaxDepth, yMinDepth, yMaxDepth, epsg_rast, lon1, lon2, lat1, lat2,
        #          xbuffer, ybuffer, if_show_outline, outline_geometry,
        #          outline_color, outline_style, outline_thickness, outline_face,
        #          outline_alpha, outline_label, outline_labelPosition,
        #          outline_labelSize, if_outline_label, alpha=0.8,
        #          if_save_fig=False, path_fig=path_fig)

        # Map depths > 0 at building level
        gdf_filtered = gdf_dmg[(gdf_dmg[stat_name] > 0)]
        if rp == max(return_periods_fig):
            depth_max = cbar_magnitude(gdf_filtered['MeanDepth'].max()*0.75)
        title = f'Mean Building Flood Depth: RP={rp}-yr {flood_data}'
        legend_label = 'Depth (m)'
        file_fig = f'{file_prefix}_{flood_data}_RP{rp}_building_depth_legend.png'
        path_fig = os.path.join(dir_figures, file_fig)
        map_vector(gdf_filtered, 'MeanDepth', outline_geometry,
                   lat1, lat2, lon1, lon2, cmap_h2o, title=title,
                   legend_label=legend_label, if_classes=False,
                   vmin=0.0, vmax=depth_max, outline_color=outline_color,
                   outline_style=outline_style,
                   outline_thickness=outline_thickness,
                   outline_face=outline_face,
                   outline_alpha=outline_alpha,
                   outline_label='Area of Interest',
                   outline_labelPosition=outline_labelPosition,
                   outline_labelSize=outline_labelSize,
                   if_show_outline=if_show_outline,
                   if_outline_label=if_outline_label,
                   xbuffer=xbuffer, ybuffer=ybuffer,
                   if_save_fig=True, path_fig=path_fig)

    # Save the updated geodataframe to a shapefile
    print(f'  Writing depths: {depth_vect}')
    gdf_dmg.to_file(depth_vect, driver='GeoJSON')

print('Done computing flood depths at each building')

### Calculating economic damage to buildings
Based on the flood water depths, the damage to the buildings (reconstruction costs) and for its contents are determined.
- First the fractional building damage is calculated applying the JRC damage functions for each classifiction (residential, commerical, etc).
- Then the fractional damage is multiplied with the maximum damage value per square meter and the building footprint area in meters and written to a shapefile.
- The damages in millions of Euros summed over all of the classes and plotted for each return period level.

In [ ]:
for rp in sorted(return_periods, reverse=True):

    depth_vect = next((s for s in depth_vects if f'RP{rp}_' in s), None)

    # Read Building Water Depth Shapefile
    gdf_dmg = gpd.read_file(depth_vect)

    print(f'Compute Building Damage for {depth_stat} Depth: RP={str(rp)}')

    stat_name = depth_stat.title() + 'Depth'

    # Compute the damage factor for each building class

    gdf_dmg['TotDamage'] = 0  # Initialize TotalDamage column
    gdf_dmg['TotDamageM2'] = 0  # Initialize TotalDamage column

    for dmg_class in dmg_classes:

        bldgDamage='f'+dmg_class[:3].upper()+depth_stat
        dmg_name = 'Dmg'+bldgDamage[1:]
        dmg_m2_name = 'Dmg'+bldgDamage[1:]+'_m2'
        # Damage factors and maximum damage value including contents
        if dmg_class == 'Residential':
            gdf_dmg[bldgDamage] = damage_function(gdf_dmg[stat_name],
                                                  coefs_res, wd_range=(0, 6))
            maxdmg = maxdmg_res.sum()
        elif dmg_class == 'Commercial':
            gdf_dmg[bldgDamage] = damage_function(gdf_dmg[stat_name],
                                                  coefs_com, wd_range=(0, 5))
            maxdmg = maxdmg_com.sum()
        elif dmg_class == 'Industrial':
            gdf_dmg[bldgDamage] = damage_function(gdf_dmg[stat_name],
                                                  coefs_ind, wd_range=(0, 5))
            maxdmg = maxdmg_ind.sum()
        elif dmg_class == 'Transportation':
            gdf_dmg[bldgDamage] = damage_function(gdf_dmg[stat_name],
                                                  coefs_trs, wd_range=(0, 3))
            maxdmg = maxdmg_trs.sum()
        elif dmg_class == 'Agriculture':
            gdf_dmg[bldgDamage] = damage_function(gdf_dmg[stat_name],
                                                  coefs_agr, wd_range=(0, 5))
            maxdmg = maxdmg_agr.sum()
        elif dmg_class == 'Universal':
            gdf_dmg[bldgDamage] = damage_function(gdf_dmg[stat_name],
                                                  coefs_uni, wd_range=(0, 6))
            maxdmg = maxdmg_uni.sum()
        else:
            gdf_dmg[bldgDamage] = damage_function(gdf_dmg[stat_name],
                                                  coefs_uni, wd_range=(0, 6))
            maxdmg = maxdmg_uni.sum()

        # Damage computation
        gdf_dmg.loc[gdf_dmg['bldgClass'] != dmg_class, bldgDamage] = 0
        gdf_dmg[dmg_name] = gdf_dmg[bldgDamage] * gdf_dmg['Area_m2'] * maxdmg
        gdf_dmg[dmg_m2_name] = gdf_dmg[bldgDamage] * maxdmg

        # Add TotalDamage in millions of €
        gdf_dmg['TotDamage'] += gdf_dmg[dmg_name] / 10**6
        gdf_dmg['TotDamageM2'] += gdf_dmg[dmg_m2_name]

    print(f'  Writing Damage: {depth_vect}')
    gdf_dmg.to_file(depth_vect, driver='GeoJSON')

    # Plotting the GeoDataFrame with filtered values
    if rp in return_periods_fig and if_building_dmg:
        file_fig = f'{file_prefix}_{flood_data}_RP{rp}_building_damage.png'
        path_fig = os.path.join(dir_figures, file_fig)
        title = f'Building Damage per Square Meter: RP={rp}-yr {flood_data}'
        legend_label = 'Damage (€/m$^2$)'
        if rp == max(return_periods_fig):
            dmg_max = cbar_magnitude(gdf_dmg['TotDamageM2'].max()*0.9)
        map_vector(gdf_dmg, 'TotDamageM2', outline_geometry,
                  lat1, lat2, lon1, lon2, 'Reds', title=title,
                  legend_label=legend_label, if_classes=False,
                  vmin=0.0, vmax=dmg_max, outline_color=outline_color,
                  outline_style=outline_style,
                  outline_thickness=outline_thickness, outline_face=outline_face,
                  outline_alpha=outline_alpha, outline_label='Area of Interest',
                  outline_labelPosition=outline_labelPosition,
                  outline_labelSize=outline_labelSize,
                  if_show_outline=if_show_outline,
                  if_outline_label=if_outline_label,
                  xbuffer=xbuffer, ybuffer=ybuffer,
                  if_save_fig=True, path_fig=path_fig)

print('Done computing damage for each building')

### Total damage to buildings
In this section the total damage for the region of interest is summed and written to a CSV file.

In [ ]:
ndmg = np.append(dmg_classes, 'Total')

df_dmg = pd.DataFrame(columns=[])
df_dmg.index = ndmg
df_dmg.index.name = 'Building Class'

for rp in return_periods:

    depth_vect = next((s for s in depth_vects if f'RP{rp}_' in s), None)
    gdf_osm = gpd.read_file(depth_vect)

    vdmg = pd.DataFrame(columns=[])
    for dmg_class in dmg_classes:
        dmg_name = f'DmgTRS{depth_stat.lower()}' if dmg_class == 'Transportation' else \
        f'Dmg{dmg_class[:3].upper()}{depth_stat.lower()}'
        gdf_osm[dmg_name] = pd.to_numeric(gdf_osm[dmg_name], errors='coerce').fillna(0)
        vdmg = np.append(vdmg, gdf_osm[dmg_name].sum())

    totdmg = sum(vdmg)
    vdmg = np.append(vdmg, totdmg)
    totdmg_byclass = pd.DataFrame(vdmg)
    # Assign names to the dataframe headers
    totdmg_byclass.columns = [f'{rp}-yr']
    totdmg_byclass.index = [ndmg]

    # Compute the total damage across the entire area of interest
    print(f'  RP={rp}-yr: Total damage (€) = {round(totdmg)}')

    df_dmg[f'{rp}-yr'] = np.array(totdmg_byclass)

damage_csv = f'{file_prefix}_{flood_data}_Damage.csv'
damage_csv = os.path.join(dir_damage, damage_csv)
df_dmg.to_csv(damage_csv)

print('Done')

### Expected Annual Damage
In this section the plot of building damages vs return periods of the flood maps is generated.
Moreover, by integrating the curve, an estimate of the expected annual damage (EAD) in millions of Euros is provided. EAD is the damage that the region would expect on average in any given year.

In [ ]:
max_yaxis=0
vert_spacer=0

# Load damage data for depth statistic
df_dmg = pd.read_csv(damage_csv, index_col=0)

# Add column with RP=1 (if it doesn't exist)
if '1-yr' not in df_dmg.columns:
    df_dmg.insert(0, '1-yr', 0)
    rps_plot = np.array([1] + return_periods)
else:
    rps_plot = np.array(return_periods)

# Compute the total Estimated Annual Damage (EAD) over all return periods
prob_rps = 1 / np.array(return_periods)
itot = df_dmg.index.get_loc('Total')
ead = 0
for irp in range(len(return_periods)-1):
    diff_rp = prob_rps[irp] - prob_rps[irp+1]
    avgdmg = (df_dmg.iloc[itot, irp+1] + df_dmg.iloc[itot, irp]) / 2
    ead = ead + avgdmg * diff_rp
    plot_text = f'{depth_stat.title()} Depth Expected Annual Damage: {round(ead/10**6, 2)} Mil €'

# Plot estimated direct damage vs exceedance probability
if if_building_dmg_plot:
    df_totdmg = df_dmg.loc['Total'] / 10**6
    max_yaxis_current=df_totdmg.max()
    max_yaxis = max(max_yaxis,max_yaxis_current)
    plt.plot(rps_plot, df_totdmg, marker='o', linestyle='-')
    plt.ylim(0)
    plt.grid(which='both', linestyle=':', linewidth=0.5, color='gray',dashes=(1,5))
    plt.xlabel('Return Period (Years)')
    plt.ylabel('Direct Damage (Mil €)')
    plt.text(0.96, 0.05+vert_spacer, plot_text,
             transform=plt.gca().transAxes, fontsize=10,
             verticalalignment='bottom', horizontalalignment='right',
             bbox=dict(facecolor='white', alpha=0.5, edgecolor='black'))
vert_spacer = vert_spacer + 0.08

yaxis_buffer=0.0001 #This sets the y axis max to be the smallest multiple larger than the max value (eg: if yaxis_buffer=100, and the max value is 280, the yaxis max will be 300)
plt.ylim(0, ((max_yaxis // yaxis_buffer) + 1) * yaxis_buffer)
plt.title(f'Annual Building Direct Damage: {flood_data}')
if if_save_fig:
    file_fig = f'{file_prefix}_{flood_data}_graph_damage.png'
    plt.savefig(os.path.join(dir_figures, file_fig), bbox_inches='tight')
plt.show()

print('Done computing expected annual damage')

### Critical Infrastructure
In order to visualise the exposure of critical infrastructure for the area of interest, the OSM dataset is used:
- Markers and colours are attributed for each type of critical infrastructure.
- Flood water depths are read.
- Critical infrastructure and floods are mapped together.

Note that preference will be given to amenity classes over building classes, to avoid duplicates. Therefore there might be cases where certain OSM entries will not show up in the map.

In [ ]:
# Ensure the critical column is boolean
gdf_osm_reclass['critInfrastructure'] = gdf_osm_reclass['critInfrastructure'].infer_objects(copy=False)

for rp in sorted(return_periods, reverse=True):
    depth_raster = next((s for s in depth_rasters if f'RP{rp}_' in s), None)

    # Read the TIFF image
    with rasterio.open(depth_raster) as src:
        r_depths = src.read(1)  # Reading the first band
        r_depths = np.ma.masked_where((r_depths < -999) | (r_depths > 1000), r_depths)
        # Compute the maximum value from the masked data
        max_depth  = r_depths.max()
        if customMaxDepthLegend == -1:
            maxDepthLegend = ((max_depth // 1) + 1)
        else:
            maxDepthLegend=customMaxDepthLegend
        missing_data_value = src.nodata

    if rp == max(return_periods_fig):
        depth_max = cbar_magnitude(max_depth*0.75)
    fig=plt.figure()
    ax = plt.axes()
    im = ax.imshow(r_depths, vmin=0, vmax=depth_max, cmap=cmap_h2o,
                   extent=(xMinDepth, xMaxDepth, yMinDepth, yMaxDepth),
                   zorder=1, alpha=0.6)
    plt.title(f'Critical Infrastructure: RP={rp}-year {flood_data}')
    divider = make_axes_locatable(ax)
    plt.xlabel('Longitude', fontsize='small')
    plt.ylabel('Latitude', fontsize='small')
    cax = divider.append_axes("right", size="2%", pad=0.1)
    plt.colorbar(im, cax=cax,label='Depth (m)',extend='max')
    plt.text(0.02,0.02,f'Projection in {gdf_dmg.crs}', fontsize=8,
             path_effects=[pe.withStroke(linewidth=3, foreground="white")],
             transform=ax.transAxes,zorder=7)

    # Collect critical buildings for all building types
    for building_type, props in crit_marker_colors.items():
        marker_item = props['marker']
        color_item = props['color']
        # Check for building type in 'amenity' or 'building' columns
        if building_type in gdf_osm_reclass['amenity'].unique():
            crit_buildings = gdf_osm_reclass[gdf_osm_reclass['amenity'] == building_type]
        elif building_type in gdf_osm_reclass['building'].unique():
            crit_buildings = gdf_osm_reclass[gdf_osm_reclass['building'] == building_type]
        else:
            continue
        crit_buildings_geometry = crit_buildings.geometry
        crit_buildings_geometry = crit_buildings_geometry.to_crs('EPSG:3857')
        crit_buildings_centroids = crit_buildings_geometry.centroid
        transformer = Transformer.from_crs('EPSG:3857', gdf_osm_reclass.crs,
                                           always_xy=True)
        reprojected_centroids = []
        for centroid in crit_buildings_centroids:
            x, y = transformer.transform(centroid.x, centroid.y)  # Reproject the coordinates
            reprojected_centroids.append(Point(x, y))
        crit_buildings_centroids_reprojected = gpd.GeoSeries(reprojected_centroids,
                                                             crs=gdf_osm_reclass.crs)
        ax.scatter(crit_buildings_centroids_reprojected.x,
                   crit_buildings_centroids_reprojected.y,
                   color=color_item, marker=marker_item, s=100,
                   linewidths=.8, edgecolors='k',
                   label=f'{building_type.capitalize()}',zorder=6) #change s value for the size of markers
    # Optionally, add basemap if needed
    ax.set_xlim(lon1-xbuffer, lon2+xbuffer)
    ax.set_ylim(lat1-ybuffer, lat2+ybuffer)
    ctx.add_basemap(ax=ax, crs=gdf_dmg.crs,
                    source=ctx.providers.OpenStreetMap.Mapnik,
                    attribution=None, alpha=0.5)
    txt = ax.texts[-1]
    txt.set_position([0.99,0.98])
    txt.set_ha('right')
    txt.set_va('top')
    #Legend
    if if_show_outline:
        outline_geometry.plot(ax=ax, edgecolor=outline_color,
                             linestyle=outline_style,
                             linewidth=outline_thickness,
                             facecolor=outline_face, alpha=outline_alpha,
                             zorder=5)
    if if_outline_label:
        outline_legend = mlines.Line2D([], [], color=outline_color,
                                       linewidth=outline_thickness,
                                       label=outline_label)
        legend_outline=ax.legend(handles=[outline_legend],
                                 loc=outline_labelPosition,
                                 fontsize=outline_labelSize, frameon=True)

    legend = ax.legend(loc='upper center', ncol=3, bbox_to_anchor=(0.5, -0.11),
                        title='Critical infrastructure type:')
    ax.axis('off')

    # Remove OSM text
    if ax.texts:
        for txt in ax.texts:
            txt.set_visible(False)

    if if_outline_label:
        ax.add_artist(legend_outline)

    if if_save_fig:
        file_fig = f'{file_prefix}_{flood_data}_RP{rp}_CriticalInfrastructure.png'
        plt.savefig(os.path.join(dir_figures, file_fig), bbox_inches='tight')
    plt.show()

print('Done with critical infrastructure exposure')

### Exposed and Displaced Population
Based on the flood depth maps, the exposed and displaced populations are estimated.
- The population and flood rasters are compared.
- The exposed and displaced populations are written to CSV files.
- Maps of the exposed and dispalced popoulations are produced.
- The exposed and displaced population are plotted against the flood map return period.

Expected annual exposed and displaced population are also calculated, representing the expected number of people exposed on average in any given year.

Please note that due to the resolution of both the population and the flood maps, it might be that part of the population appears to be over a water body (eg: a river) and is counted towards the overall exposed statistics.

### Resample population data to flood grid

In [ ]:
print('  Loading and interpolating population data to flood raster grid:')
pop_raster = os.path.join(dir_pop, f'{file_prefix}_pop_{file_pop_string}.tif')
print(f'  {pop_raster}')

# Get target grid parameters
depth_da = rio.open_rasterio(depth_rasters[0]).squeeze('band', drop=True)
target_crs = depth_da.rio.crs
target_transform = depth_da.rio.transform()
target_shape = depth_da.rio.shape

# Load clipping polygon
bounding_poly = gpd.read_file(path_vect_bbox)

if pop_data == 'GHSL':
    with rio.open_rasterio(pop_full_raster) as src:
        src = src.squeeze('band', drop=True)
        bounding_poly_src = bounding_poly.to_crs(src.rio.crs)
        clipped = src.rio.clip(bounding_poly_src.geometry,
                              all_touched=True,
                              from_disk=True)
        pop_reprojected = clipped.rio.reproject(
            target_crs,
            shape=target_shape,
            transform=target_transform,
            resampling=Resampling.bilinear
        )
elif pop_data == 'SEDAC':
    with xr.open_dataset(pop_full_raster) as ds:
        pop = ds['Band1'].rio.set_spatial_dims('lon', 'lat')
        pop.rio.write_crs("EPSG:4326", inplace=True)
        bounding_poly_4326 = bounding_poly.to_crs("EPSG:4326")
        clipped = pop.rio.clip(bounding_poly_4326.geometry,
                              all_touched=True)
        pop_reprojected = clipped.rio.reproject(
            target_crs,
            shape=target_shape,
            transform=target_transform,
            resampling=Resampling.bilinear
        )

assert pop_reprojected.rio.transform() == target_transform, \
    "Final transform mismatch!"
assert pop_reprojected.rio.shape == target_shape, \
    "Final shape mismatch!"

# Write output
pop_reprojected.rio.to_raster(
    pop_raster,
    dtype='float32',
    compress='DEFLATE'
)

print(f'Population data aligned: {pop_raster}')

### Compute exposed and displaced population for each return period

In [ ]:
pop_exposed_rasts = []
pop_displaced_rasts = []
df_pop_exposed = []
df_pop_displaced = []
rps = []

# Process each return period
for rp in sorted(return_periods, reverse=True):

    depth_raster = next((s for s in depth_rasters if f'RP{rp}_' in s), None)

    pop_exposed_rast = os.path.join(dir_pop, f'{file_prefix}_{flood_data}_RP{rp}_exposed_pop_{file_pop_string}.tif')
    pop_exposed_rasts.append(pop_exposed_rast)
    pop_displaced_rast = pop_exposed_rast.replace('exposed', 'displaced')
    pop_displaced_rasts.append(pop_displaced_rast)

    # Read both rasters (assuming same CRS, transform, and grid)
    with rio.open_rasterio(depth_raster) as depth_da, rio.open_rasterio(pop_raster) as pop_da:

        # Squeeze single-band dimensions
        rdepth = depth_da.squeeze('band', drop=True)
        rpop = pop_da.squeeze('band', drop=True)

        # Calculate population exposure
        exp_mask = rdepth > depth_min_exposed
        exposed_pop = (exp_mask * rpop).where((exp_mask * rpop) >= 0, np.nan)
        # Calculate population displaced
        exp_mask = rdepth > depth_min_displaced
        #displaced_pop = exp_mask*rpop
        displaced_pop = (exp_mask * rpop).where((exp_mask * rpop) >= 0, np.nan)

        # Write
        exposed_pop.rio.to_raster(pop_exposed_rast)
        displaced_pop.rio.to_raster(pop_displaced_rast)
        print(pop_exposed_rast)

        # Statistics
        rps.append(rp)
        total_exposed = float(exposed_pop.sum().item())
        df_pop_exposed.append(total_exposed)
        total_displaced = float(displaced_pop.sum().item())
        df_pop_displaced.append(total_displaced)

    # Plot rasters
    if rp in return_periods_fig and If_population_exp:
        file_fig = f'{file_prefix}_RP{rp}_map_exposed_pop_{file_pop_string}.png'
        path_fig = os.path.join(dir_figures, file_fig)
        title = f'Population Exposed: RP={rp}-yr {title_pop_string} {flood_data}'
        legend_label = 'People'
        text_label = f'Exposed if Water Depth >{depth_min_exposed}m'
        x_label, y_label = 0.0, 0.0
        exposed_pop = np.ma.masked_where((exposed_pop <= 0), exposed_pop)
        if rp == max(return_periods_fig):
            pop_max = cbar_magnitude(np.nanmax(exposed_pop)*0.9)
        map_raster(exposed_pop, title, legend_label, 0, pop_max, cmap_pop,
                   lon1, lon2, lat1, lat2, 'upper', epsg_rast,
                   lon1, lon2, lat1, lat2,
                   xbuffer, ybuffer, if_show_outline, outline_geometry,
                   outline_color, outline_style, outline_thickness, outline_face,
                   outline_alpha, outline_label, outline_labelPosition,
                   outline_labelSize, if_outline_label,
                   text_label=text_label, x_label=x_label, y_label=y_label,
                   alpha=0.7, if_save_fig=if_save_fig, path_fig=path_fig)

        file_fig = f'{file_prefix}_RP{rp}_map_displaced_pop_{file_pop_string}.png'
        path_fig = os.path.join(dir_figures, file_fig)
        title = f'Population Displaced: RP={rp}-yr {title_pop_string} {flood_data}'
        legend_label = 'People'
        text_label = f'Displaced if Water Depth >{depth_min_displaced}m'
        x_label, y_label = 0.0, 0.0
        displaced_pop = np.ma.masked_where((displaced_pop <= 0), displaced_pop)
        if rp == max(return_periods_fig):
            pop_max = 0 if np.isnan(displaced_pop).all() else cbar_magnitude(np.nanmax(displaced_pop) * 0.9)
        map_raster(displaced_pop, title, legend_label, 0, pop_max, cmap_pop,
                   lon1, lon2, lat1, lat2, 'upper', epsg_rast,
                   lon1, lon2, lat1, lat2,
                   xbuffer, ybuffer, if_show_outline, outline_geometry,
                   outline_color, outline_style, outline_thickness, outline_face,
                   outline_alpha, outline_label, outline_labelPosition,
                   outline_labelSize, if_outline_label,
                   text_label=text_label, x_label=x_label, y_label=y_label,
                   alpha=0.7, if_save_fig=if_save_fig, path_fig=path_fig)

print(df_pop_exposed)

### Compute estimated annual exposed and displaced population (EAEP and EADP) over all return periods

In [ ]:
eaep = 0
eadp = 0
for irp in range(len(return_periods)-1):
    diff_rp = prob_rps[irp] - prob_rps[irp+1]
    avgpop = (df_pop_exposed[irp+1] + df_pop_exposed[irp]) / 2
    eaep = eaep + avgpop * diff_rp
    avgpop = (df_pop_displaced[irp+1] + df_pop_displaced[irp]) / 2
    eadp = eadp + avgpop * diff_rp

if If_population_exp_plot:
    plt.plot(np.array([1] + return_periods), [0] + sorted(df_pop_exposed), marker='o', linestyle='-')
    plt.ylim(0)
    plt.grid(which='both', linestyle=':', linewidth=0.5, color='gray',dashes=(1,5))
    plt.xlabel('Return Period (Years)')
    plt.ylabel('People')
    plt.title(f'Estimated Exposed Population: {title_pop_string} {flood_data}')
    plot_text = f'Expected Annual Population Exposed: {round(eaep)} people.'
    plt.text(0.96, 0.05, plot_text, transform=plt.gca().transAxes, fontsize=10,
             verticalalignment='bottom', horizontalalignment='right',
             bbox=dict(facecolor='white', alpha=0.5, edgecolor='black'))
    if if_save_fig:
        file_fig = f'{file_prefix}_{flood_data}_graph_exposed_pop_{file_pop_string}.png'
        plt.savefig(os.path.join(dir_figures, file_fig), bbox_inches='tight')
    plt.show()

    plt.plot(np.array([1] + return_periods), [0] + sorted(df_pop_displaced), marker='o', linestyle='-')
    plt.ylim(0)
    plt.grid(which='both', linestyle=':', linewidth=0.5, color='gray',dashes=(1,5))
    plt.xlabel('Return Period (Years)')
    plt.ylabel('People')
    plt.title(f'Estimated Displaced Population: {title_pop_string} {flood_data}')
    plot_text = f'Expected Annual Population Displaced: {round(eadp)} people.'
    plt.text(0.96, 0.05, plot_text, transform=plt.gca().transAxes, fontsize=10,
             verticalalignment='bottom', horizontalalignment='right',
             bbox=dict(facecolor='white', alpha=0.5, edgecolor='black'))

    if if_save_fig:
        file_fig = f'{file_prefix}_{flood_data}_graph_displaced_pop_{file_pop_string}.png'
        plt.savefig(os.path.join(dir_figures, file_fig), bbox_inches='tight')
    plt.show()


# csv files
df_pop = pd.DataFrame(columns=[])
df_pop.index = rps
df_pop.index.name = 'Return Period (years)'
df_pop['People Exposed'] = df_pop_exposed
df_pop = df_pop.sort_index()
pop_csv = f'{file_prefix}_{flood_data}_exposed_pop_{file_pop_string}.csv'
pop_csv = os.path.join(dir_pop, pop_csv)
df_pop.to_csv(pop_csv)
print(df_pop)

df_pop = pd.DataFrame(columns=[])
df_pop.index = rps
df_pop.index.name = 'Return Period (years)'
df_pop['People Displaced'] = df_pop_displaced
df_pop = df_pop.sort_index()
pop_csv = f'{file_prefix}_{flood_data}_displaced_pop_{file_pop_string}.csv'
pop_csv = os.path.join(dir_pop, pop_csv)
df_pop.to_csv(pop_csv)
print(df_pop)

print('\nDone computing exposed and displaced populations')

### Resample Crop Data to Flood Grid
- This include summing rainfed and irrigated crop fractions
- Bilinear interpolation

In [ ]:
# Open Landuse NetCDF
ds = xr.open_dataset(lu_full_nc).sel(
    latitude=slice(lat2+0.25, lat1-0.25),
    longitude=slice(lon1-0.25, lon2+0.25))
#xr_crop_rf = ds['crop_rf'].where(ds['crop_rf'] > 0) / 100.0

# Load rainfed and irrigated crop
xr_crop_rf = ds['crop_rf'] / 100.0
xr_crop_irr = ds['crop_irr'] / 100.0

# Sum and convert from percent to fraction
xr_crop = xr_crop_rf + xr_crop_irr

# Set spatial parameters for resampling
xr_crop = xr_crop.rio.set_spatial_dims('longitude', 'latitude')
xr_crop.rio.write_crs("EPSG:4326", inplace=True)

# Open flood raster
depth_da = rio.open_rasterio(depth_rasters[0]).squeeze('band', drop=True)

# Resample crop data to depth_raster
xr_crop = xr_crop.rio.set_spatial_dims('longitude', 'latitude')
xr_crop.rio.write_crs("EPSG:4326", inplace=True)
crop_resampled = xr_crop.rio.reproject_match(depth_da, resampling=Resampling.bilinear)

# Load region area of clipping
bounding_poly = gpd.read_file(path_vect_bbox)
bounding_poly = bounding_poly.to_crs(crop_resampled.rio.crs)

# Perform clipping
crop_resampled = crop_resampled.rio.clip(
    bounding_poly.geometry,  # Geometry to clip with
    all_touched=True,
    drop=True  # Drop pixels outside the geometry
)

lu_rast = os.path.join(dir_lu, f'{file_prefix}_{os.path.basename(lu_full_nc).replace(".nc", ".tif")}')
crop_resampled.rio.to_raster(lu_rast)

path_fig = os.path.join(dir_figures, os.path.basename(lu_rast).replace(".tif", ".png"))
title = f'Resampled Crop Fraction: {lu_ssp} {lu_year}'
legend_label = 'fraction'
map_raster(crop_resampled, title, legend_label, 0.0, 1.0, cmap_h2o,
           lon1, lon2, lat1, lat2, 'upper', epsg_rast, lon1, lon2, lat1, lat2,
           xbuffer, ybuffer, True, outline_geometry,
           outline_color, outline_style, outline_thickness, outline_face,
           outline_alpha, outline_label, outline_labelPosition,
           outline_labelSize, if_outline_label, alpha=0.7,
           if_save_fig=False, path_fig=path_fig)

print(f'Crop data interpolated to flood grid:\n{lu_rast}')

### Calculate crop area exposed

In [ ]:
# Open Resampled Crop Raster on Flood Grid
crop_frac = rio.open_rasterio(lu_rast).squeeze('band', drop=True)

# Empty lists
df_crop_exposed = []
crop_exposed_rasts = []
df_crop_dmg = []
crop_dmg_rasts = []
rps = []

# Process each return period
for rp in sorted(return_periods, reverse=True):

    rps.append(rp)

    depth_raster = next((s for s in depth_rasters if f'RP{rp}_' in s), None)

    # Load flood depth raster
    depth_da = rio.open_rasterio(depth_raster).squeeze('band', drop=True)

    # Cell area in m²
    if rp == max(return_periods):
        transform = depth_da.rio.transform()
        dy, dx = abs(transform.a), abs(transform.e)
        if depth_da.rio.crs.is_geographic:
            lat_rad = np.deg2rad(depth_da.y.values)
            cell_area = (6371e3 ** 2) * np.cos(lat_rad) * (dx * np.pi/180) * (dy * np.pi/180)
            area_m2 = np.tile(cell_area[:, np.newaxis], (1, depth_da.x.size))
        else:
            area_m2 = np.full((depth_da.y.size, depth_da.x.size), dx * dy)
        area_m2 = xr.DataArray(area_m2, dims=["y", "x"], coords={"y": depth_da.y, "x": depth_da.x})

    # Crop area exposed
    exposed_crop = (depth_da > depth_min_exposed) * (crop_frac * area_m2)
    exposed_crop = exposed_crop.rio.write_crs(depth_da.rio.crs)

    # Write to exposed crop area to raster
    crop_exposed_rast = f'{file_prefix}_{flood_data}_RP{rp}_crop_exposed_{lu_ssp}_{lu_year}.tif'
    crop_exposed_rast = os.path.join(dir_lu, crop_exposed_rast)
    crop_exposed_rasts.append(crop_exposed_rast)
    exposed_crop.rio.to_raster(crop_exposed_rast)

    # Sum the exposed crop area in km2
    total_exposed = np.sum(exposed_crop).item() / 1e6
    df_crop_exposed.append(total_exposed)


    # Depth for 100% damage (when damage function values hit 1)
    depth100 = wd_values[np.argmax(np.array(dmg_agr) >= 1.0)]

    # Crop damage
    crop_dmg = np.where(
        depth_da < depth100,
        (coefs_agr[0] * depth_da**5 +
         coefs_agr[1] * depth_da**4 +
         coefs_agr[2] * depth_da**3 +
         coefs_agr[3] * depth_da**2 +
         coefs_agr[4] * depth_da +
         coefs_agr[5]).clip(0, 1),
        1
    )
    crop_dmg = crop_dmg * crop_frac * area_m2 * maxdmg_agr.sum()

    # Write to crop damage to raster
    crop_dmg_rast = f'{file_prefix}_{flood_data}_RP{rp}_crop_damage_{lu_ssp}_{lu_year}.tif'
    crop_dmg_rast = os.path.join(dir_lu, crop_dmg_rast)
    crop_dmg_rasts.append(crop_dmg_rast)
    crop_dmg.rio.to_raster(crop_dmg_rast)

    # Sum the crop damage in Euros
    crop_total_dmg = np.sum(crop_dmg).item() / 1e6
    df_crop_dmg.append(crop_total_dmg)

    if rp in return_periods_fig and If_population_exp:
        # Plot rasters
        height, width = exposed_crop.shape
        xMinExpCrop, yMinExpCrop, xMaxExpCrop, yMaxExpCrop = array_bounds(height, width, out_transform)
        file_fig = f'{file_prefix}_RP{rp}_map_exposed_crop_{lu_ssp}_{lu_year}.png'
        path_fig = os.path.join(dir_figures, file_fig)
        title = f'Crop Exposed: {lu_ssp} {lu_year}: RP={rp}-yr {flood_data}'
        legend_label = 'Area (m²)'
        text_label = f'Exposed if Water Depth >{depth_min_exposed}m'
        x_label, y_label = 0.0, 0.0
        crop_exposed = np.ma.masked_where((exposed_crop <= 0), exposed_crop)
        if rp == max(return_periods_fig):
            crop_max = cbar_magnitude(np.nanmax(crop_exposed)*0.9)
        map_raster(crop_exposed, title, legend_label, 0, crop_max, cmap_pop,
                   lon1, lon2, lat1, lat2, 'upper', epsg_rast,
                   lon1, lon2, lat1, lat2,
                   xbuffer, ybuffer, if_show_outline, outline_geometry,
                   outline_color, outline_style, outline_thickness, outline_face,
                   outline_alpha, outline_label, outline_labelPosition,
                   outline_labelSize, if_outline_label,
                   text_label=text_label, x_label=x_label, y_label=y_label,
                   alpha=0.7, if_save_fig=if_save_fig, path_fig=path_fig)

        # Plot rasters
        height, width = exposed_crop.shape
        xMinExpCrop, yMinExpCrop, xMaxExpCrop, yMaxExpCrop = array_bounds(height, width, out_transform)
        file_fig = f'{file_prefix}_RP{rp}_map_exposed_crop_{lu_ssp}_{lu_year}.png'
        path_fig = os.path.join(dir_figures, file_fig)
        title = f'Crop Damage {lu_ssp} {lu_year}: RP={rp}-yr {flood_data}'
        legend_label = 'Damage (€)'
        x_label, y_label = 0.0, 0.0
        crop_dmg = np.ma.masked_where((crop_dmg <= 0), crop_dmg)
        if rp == max(return_periods_fig):
            crop_dmg_max = cbar_magnitude(np.nanmax(crop_dmg)*0.9)
        map_raster(crop_dmg, title, legend_label, 0, crop_dmg_max, cmap_pop,
                   lon1, lon2, lat1, lat2, 'upper', epsg_rast,
                   lon1, lon2, lat1, lat2,
                   xbuffer, ybuffer, if_show_outline, outline_geometry,
                   outline_color, outline_style, outline_thickness, outline_face,
                   outline_alpha, outline_label, outline_labelPosition,
                   outline_labelSize, if_outline_label,
                   text_label=text_label, x_label=x_label, y_label=y_label,
                   alpha=0.7, if_save_fig=if_save_fig, path_fig=path_fig)

### Compute the total Estimated Annual Exposed Crop (EAEC) and damaged crop (EACD) over all return periods

In [ ]:

eaec = 0
eacd = 0
prob_rps = 1 / np.array(return_periods)
for irp in range(len(return_periods)-1):
    diff_rp = prob_rps[irp] - prob_rps[irp+1]
    avgcrop = ((df_crop_exposed[irp+1] + df_crop_exposed[irp]) / 2)
    eaec = eaec + avgcrop * diff_rp
    avgcrop = ((df_crop_dmg[irp+1] + df_crop_dmg[irp]) / 2)
    eacd = eacd + avgcrop * diff_rp

if If_population_exp_plot:
    plt.plot(np.array([1] + return_periods), [0] + sorted(df_crop_exposed), marker='o', linestyle='-')
    plt.ylim(0)
    plt.grid(which='both', linestyle=':', linewidth=0.5, color='gray',dashes=(1,5))
    plt.xlabel('Return Period (Years)')
    plt.ylabel('Crop Area')
    plt.title(f'Estimated Exposed Crop: {lu_ssp} {lu_year} {flood_data}')
    plot_text = f'Expected Annual Exposed Crop: {round(eaec, 2)} km²'
    plt.text(0.96, 0.05, plot_text, transform=plt.gca().transAxes, fontsize=10,
             verticalalignment='bottom', horizontalalignment='right',
             bbox=dict(facecolor='white', alpha=0.5, edgecolor='black'))

    if if_save_fig:
        file_fig = f'{file_prefix}_{flood_data}_graph_exposed_crop_{lu_ssp}_{lu_year}.png'
        plt.savefig(os.path.join(dir_figures, file_fig), bbox_inches='tight')
    plt.show()

    plt.plot(np.array([1] + return_periods), [0] + sorted(df_crop_dmg), marker='o', linestyle='-')
    plt.ylim(0)
    plt.grid(which='both', linestyle=':', linewidth=0.5, color='gray',dashes=(1,5))
    plt.xlabel('Return Period (Years)')
    plt.ylabel('Crop Damage')
    plt.title(f'Estimated Crop Damage {lu_ssp} {lu_year} {flood_data}')
    plot_text = f'Expected Annual Crop Damage: {round(eacd, 2)} M€'
    plt.text(0.96, 0.05, plot_text, transform=plt.gca().transAxes, fontsize=10,
             verticalalignment='bottom', horizontalalignment='right',
             bbox=dict(facecolor='white', alpha=0.5, edgecolor='black'))

    if if_save_fig:
        file_fig = f'{file_prefix}_{flood_data}_graph_exposed_crop_{lu_ssp}_{lu_year}.png'
        plt.savefig(os.path.join(dir_figures, file_fig), bbox_inches='tight')
    plt.show()

#CSV FILE
df_crop = pd.DataFrame(columns=[])
df_crop.index = rps
df_crop.index.name = 'Return Period (years)'
df_crop['Crop Area Exposed'] = df_crop_exposed
df_crop = df_crop.sort_index()
crop_csv = f'{file_prefix}_{flood_data}_exposed_crop_{lu_ssp}_{lu_year}.csv'
crop_csv = os.path.join(dir_lu, crop_csv)
df_crop.to_csv(crop_csv)
print(df_crop)

df_crop = pd.DataFrame(columns=[])
df_crop.index = rps
df_crop.index.name = 'Return Period (years)'
df_crop['Crop Damage (M€)'] = df_crop_dmg
df_crop = df_crop.sort_index()
crop_csv = f'{file_prefix}_{flood_data}_damage_crop_{lu_ssp}_{lu_year}.csv'
crop_csv = os.path.join(dir_lu, crop_csv)
df_crop.to_csv(crop_csv)
print(df_crop)

print('Done computing exposed crop')

## Conclusions
In the risk assessment:
- Flood and population maps where generated.
- By combining hazards with exposure and vulnerabilities the risk was computed
  - Building damage maps and estimated yearly damage.
  - Population displacement and estimated yearly displacement.
- Moreover, overall population and critical infrastructure exposed were also showcased.

## Class Reflections:
Having knowledge of the location can help assess if the results match with the expectations.
* How do the results match with your expectations?
* Are data limitations a factor or are there other reasons?

As mentioned, there are [multiple limitations](#Limitations) that should be considered and are context-dependent. Each limitation can have minimal impacts on the results for your area, or alternatively, can be a importance source of error. It is important to understand how these limitations can affect the results, and if its necessary to improve the datasets.

### Authors
CMCC

Main contributors:  
Davide Serrao, Margherita Sarcinella, Arthur Hrast Essenfelder, Jeremy Pal